# SemCons-IDS — Post-Hoc Explainability Continuation v11

Cette extension poursuit la campagne **sans réentraîner aucun détecteur**.

Elle recharge les 12 runs MiniLM/ModernBERT déjà sauvegardés et ajoute :

- baseline aléatoire Top-k ;
- courbes de suppression ;
- AOPC ;
- suffisance ;
- stabilité sous perturbations ;
- analyse par classe ;
- analyse des erreurs ;
- complémentarité XGBoost–SLM ;
- relation entre explicabilité et ensembles conformes ;
- latence TreeSHAP et occlusion ;
- verbalisation structurée, validation automatique et fallback déterministe ;
- graphiques et tableaux article-ready ;
- reprise automatique après interruption.

## 0 — Installation compatible Google Colab

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as metadata

PINNED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}

def version_of(name: str):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def stack_ok() -> bool:
    code = (
        "from transformers import AutoTokenizer, AutoModelForSequenceClassification;"
        "from peft import LoraConfig, TaskType, get_peft_model;"
        "from xgboost import XGBClassifier;"
        "print('STACK_OK')"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        capture_output=True,
    )
    return result.returncode == 0 and "STACK_OK" in result.stdout

installed = {name: version_of(name) for name in PINNED}
torchao_version = version_of("torchao")

needs_repair = (
    torchao_version is not None
    or any(installed[name] != expected for name, expected in PINNED.items())
    or not stack_ok()
)

print("Versions avant vérification :")
for name, expected in PINNED.items():
    print(f"  {name:14s}: {installed[name]} (attendu : {expected})")
print(f"  {'torchao':14s}: {torchao_version or 'absent'}")

if needs_repair:
    print("\nRéinstallation propre de la pile...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "transformers", "peft", "accelerate", "tokenizers", "torchao", "xgboost",
        ],
        check=False,
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--no-cache-dir",
            "transformers==4.57.6",
            "peft==0.18.0",
            "accelerate==1.12.0",
            "xgboost==2.1.4",
            "pyarrow>=15.0.0",
            "imbalanced-learn>=0.12.0",
            "joblib>=1.3.0",
            "psutil>=5.9.0",
            "matplotlib>=3.8.0",
            "seaborn>=0.13.2",
            "scipy>=1.11.0",
        ]
    )
    if not stack_ok():
        raise RuntimeError("La pile réinstallée n'est pas importable.")
    print("✅ Installation réparée.")
    print("♻️ Redémarrage automatique du noyau...")
    os.kill(os.getpid(), signal.SIGKILL)

print("✅ Environnement stable et prêt.")

## 1 — Imports, configuration et dossiers de sortie

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import importlib.metadata as metadata
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from xgboost import XGBClassifier

try:
    from scipy.stats import wilcoxon, spearmanr
except Exception:
    wilcoxon = None
    spearmanr = None

warnings.filterwarnings("ignore")
sns.set_context("talk")

EXPECTED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}
for package, expected in EXPECTED.items():
    found = metadata.version(package)
    if found != expected:
        raise RuntimeError(
            f"{package}={found}; attendu={expected}. "
            "Réexécutez la cellule d'installation et laissez le noyau redémarrer."
        )

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Dataset/CIC-IDS-2017/"
    output_dir: str = "/content/drive/MyDrive/these_llm/SemCons_IDS_MiniLM_ModernBERT_v8/"
    model_checkpoint: str = "sentence-transformers/all-MiniLM-L6-v2"
    current_backbone: str = "minilm"
    backbones: List[str] = field(default_factory=lambda: ["minilm", "modernbert"])
    lora_target_modules: List[str] = field(default_factory=lambda: ["query", "value"])

    seeds: List[int] = field(default_factory=lambda: [42, 123, 2026])
    protocols: List[str] = field(default_factory=lambda: ["iid_stratified", "temporal_within_file"])

    train_fraction: float = 0.70
    val_fraction: float = 0.10
    cal_fraction: float = 0.10
    test_fraction: float = 0.10

    min_samples_per_class: int = 200
    numeric_top_k: int = 60
    text_top_k: int = 16
    feature_selection_rows_per_class: int = 20_000

    llm_train_cap_per_class: int = 20_000
    llm_val_cap_per_class: int = 5_000
    max_length: int = 320

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05

    epochs: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    batch_train: int = 32
    batch_eval: int = 128
    grad_accum: int = 1
    warmup_fraction: float = 0.06
    label_smoothing: float = 0.01
    early_stopping_patience: int = 2

    use_sqrt_class_weights: bool = True
    class_weight_min: float = 0.60
    class_weight_max: float = 2.50

    xgb_n_estimators: int = 1200
    xgb_max_depth: int = 12
    xgb_learning_rate: float = 0.08
    xgb_subsample: float = 0.90
    xgb_colsample_bytree: float = 0.90
    xgb_min_child_weight: float = 1.0
    xgb_reg_lambda: float = 1.0
    xgb_early_stopping_rounds: int = 50

    fusion_selection_metric: str = "weighted_f1"
    fusion_min_semantic_weight: float = 0.00
    fusion_grid_points: int = 41

    target_weighted_f1: float = 0.99
    selection_gain_margin: float = 5e-4

    conformal_alpha: float = 0.10
    raps_lambda: float = 0.02
    raps_k_reg: int = 2
    conformal_abstain_size: int = 2

    explanation_model_checkpoint: str = "google/flan-t5-small"
    explanation_samples: int = 60
    slm_explanation_samples: int = 30
    explanation_top_k: int = 5
    explanation_max_new_tokens: int = 96
    explanation_language: str = "English"

    figure_dpi: int = 220
    resume_completed_runs: bool = True
    force_rerun: bool = False

    allow_detector_training: bool = False
    posthoc_subdir: str = "posthoc_explainability_v11"
    posthoc_resume: bool = True

    posthoc_samples_per_scenario: int = 120
    posthoc_error_quota: int = 30
    posthoc_checkpoint_every: int = 10

    deletion_k_values: List[int] = field(
        default_factory=lambda: [1, 2, 3, 5, 8, 10]
    )
    random_baseline_repeats: int = 10

    stability_samples: int = 30
    stability_repeats: int = 3
    stability_noise_fraction: float = 0.03

    verbalizer_samples: int = 40
    verbalizer_retry: bool = True

CFG = Config()


BACKBONE_SPECS = {
    "minilm": {
        "display_name": "MiniLM",
        "checkpoint": "sentence-transformers/all-MiniLM-L6-v2",
        "target_modules": ["query", "value"],
        "modules_to_save": ["classifier"],
        "batch_train": 32,
        "batch_eval": 128,
        "grad_accum": 1,
        "expected_parameters_m": 22.7,
    },
    "modernbert": {
        "display_name": "ModernBERT",
        "checkpoint": "answerdotai/ModernBERT-base",
        "target_modules": ["Wqkv"],
        "modules_to_save": ["classifier"],
        "batch_train": 16,
        "batch_eval": 64,
        "grad_accum": 2,
        "expected_parameters_m": 149.0,
    },
}

def apply_backbone_config(backbone: str) -> Dict[str, object]:
    if backbone not in BACKBONE_SPECS:
        raise KeyError(f"Backbone inconnu : {backbone}")
    spec = BACKBONE_SPECS[backbone]
    CFG.current_backbone = backbone
    CFG.model_checkpoint = str(spec["checkpoint"])
    CFG.lora_target_modules = list(spec["target_modules"])
    CFG.batch_train = int(spec["batch_train"])
    CFG.batch_eval = int(spec["batch_eval"])
    CFG.grad_accum = int(spec["grad_accum"])
    return spec

def backbone_display_name(backbone: str) -> str:
    return str(BACKBONE_SPECS[backbone]["display_name"])

apply_backbone_config(CFG.backbones[0])


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.seeds[0])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive non monté :", exc)

OUTPUT_DIR = Path(CFG.output_dir)
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
POSTHOC_DIR = OUTPUT_DIR / CFG.posthoc_subdir
POSTHOC_FIG_DIR = POSTHOC_DIR / "figures"
POSTHOC_TABLE_DIR = POSTHOC_DIR / "tables"
POSTHOC_CACHE_DIR = POSTHOC_DIR / "cache"
for directory in [
    OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR,
    POSTHOC_DIR, POSTHOC_FIG_DIR,
    POSTHOC_TABLE_DIR, POSTHOC_CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

plt.rcParams["figure.dpi"] = CFG.figure_dpi
plt.rcParams["savefig.dpi"] = CFG.figure_dpi

def save_figure(filename_stem: str) -> None:
    png = FIG_DIR / f"{filename_stem}.png"
    pdf = FIG_DIR / f"{filename_stem}.pdf"
    plt.tight_layout()
    plt.savefig(png, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Figure sauvegardée :", png)

def save_posthoc_figure(filename_stem: str) -> None:
    png = POSTHOC_FIG_DIR / f"{filename_stem}.png"
    pdf = POSTHOC_FIG_DIR / f"{filename_stem}.pdf"
    plt.tight_layout()
    plt.savefig(png, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Figure post-hoc sauvegardée :", png)


## 2 — Chargement, normalisation des labels et déduplication

In [ ]:
def normalize_column_name(name: str) -> str:
    value = str(name).strip().lower()
    value = re.sub(r"[\s/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def normalize_label(value: object) -> str | None:
    raw = str(value).strip()
    key = re.sub(r"\s+", " ", raw.lower().replace("_", " ")).strip()
    mapping = {
        "benign": "BENIGN",
        "bot": "Bot",
        "ddos": "DDoS",
        "dos hulk": "DoS_Hulk",
        "dos goldeneye": "DoS_GoldenEye",
        "dos slowhttptest": "DoS_Slowhttptest",
        "dos slowloris": "DoS_Slowloris",
        "ftp-patator": "FTP-Patator",
        "ftp patator": "FTP-Patator",
        "ssh-patator": "SSH-Patator",
        "ssh patator": "SSH-Patator",
        "portscan": "PortScan",
        "port scan": "PortScan",
        "infiltration": "Infiltration",
        "heartbleed": "Heartbleed",
    }
    return mapping.get(key)

def find_data_files(data_dir: str) -> List[Path]:
    root = Path(data_dir)
    files = []
    for pattern in ("*.parquet", "*.csv", "*.feather"):
        files.extend(root.glob(pattern))
    return sorted(files)

def read_one_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix == ".feather":
        frame = pd.read_feather(path)
    elif suffix == ".csv":
        frame = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(path)
    frame.columns = [normalize_column_name(c) for c in frame.columns]
    frame["__source_file__"] = path.name
    frame["__row_in_file__"] = np.arange(len(frame))
    return frame

files = find_data_files(CFG.data_dir)
if not files:
    raise FileNotFoundError(f"Aucun CSV/Parquet/Feather dans {CFG.data_dir}")

frames = []
for file_path in files:
    frame = read_one_file(file_path)
    frames.append(frame)
    print(f"{file_path.name:<70s} {len(frame):>10,} lignes")

df = pd.concat(frames, ignore_index=True)

label_candidates = [c for c in df.columns if c == "label" or "label" in c or "class" in c]
if not label_candidates:
    raise ValueError("Colonne label introuvable.")
LABEL_COL = label_candidates[0]

df[LABEL_COL] = df[LABEL_COL].map(normalize_label)
unmapped = int(df[LABEL_COL].isna().sum())
df = df.dropna(subset=[LABEL_COL]).copy()

before_dedup = len(df)
dedup_columns = [c for c in df.columns if c not in {"__source_file__", "__row_in_file__"}]
df = df.drop_duplicates(subset=dedup_columns).reset_index(drop=True)

feature_candidates = [
    c for c in df.columns
    if c not in {LABEL_COL, "__source_file__", "__row_in_file__"}
]
for column in feature_candidates:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[feature_candidates] = df[feature_candidates].replace([np.inf, -np.inf], np.nan)

counts = df[LABEL_COL].value_counts()
keep_classes = counts[counts >= CFG.min_samples_per_class].index.tolist()
removed_classes = sorted(set(counts.index) - set(keep_classes))
df = df[df[LABEL_COL].isin(keep_classes)].reset_index(drop=True)

print(f"\nLabels non reconnus supprimés : {unmapped:,}")
print(f"Doublons supprimés            : {before_dedup - len(df):,}")
print(f"Dataset retenu                : {len(df):,} lignes × {len(feature_candidates)} variables candidates")
print("Classes supprimées            :", removed_classes)
display(df[LABEL_COL].value_counts().to_frame("effectif"))

ALL_CLASS_COUNTS = df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
ALL_CLASS_COUNTS.to_csv(TABLE_DIR / "overall_class_distribution.csv", index=False)

## 3 — Graphe global : distribution des classes

In [ ]:
distribution_plot_df = ALL_CLASS_COUNTS.sort_values("count", ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(distribution_plot_df["label"], distribution_plot_df["count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Nombre de flux")
plt.title("Distribution globale des classes — CICIDS2017 nettoyé")
save_figure("global_class_distribution")
plt.show()

## 4 — Fonctions de split, prétraitement, MiniLM et XGBoost

In [ ]:
def make_splits(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    label_col: str,
    train_fraction: float,
    val_fraction: float,
    cal_fraction: float,
    test_fraction: float,
) -> Dict[str, pd.DataFrame]:
    if not math.isclose(train_fraction + val_fraction + cal_fraction + test_fraction, 1.0, rel_tol=0, abs_tol=1e-9):
        raise ValueError("Les fractions de split doivent sommer à 1.0.")

    if protocol == "iid_stratified":
        train_df, temp_df = train_test_split(
            data,
            test_size=1.0 - train_fraction,
            stratify=data[label_col],
            random_state=seed,
        )
        relative_val = val_fraction / (val_fraction + cal_fraction + test_fraction)
        val_df, remain_df = train_test_split(
            temp_df,
            test_size=1.0 - relative_val,
            stratify=temp_df[label_col],
            random_state=seed + 1,
        )
        relative_cal = cal_fraction / (cal_fraction + test_fraction)
        cal_df, test_df = train_test_split(
            remain_df,
            test_size=1.0 - relative_cal,
            stratify=remain_df[label_col],
            random_state=seed + 2,
        )
        splits = {
            "train": train_df.reset_index(drop=True),
            "val": val_df.reset_index(drop=True),
            "cal": cal_df.reset_index(drop=True),
            "test": test_df.reset_index(drop=True),
        }
        return splits

    if protocol == "temporal_within_file":
        buckets = {"train": [], "val": [], "cal": [], "test": []}
        for source_file, group in data.groupby("__source_file__", sort=False):
            group = group.sort_values("__row_in_file__").reset_index(drop=True)
            n = len(group)

            train_end = int(round(train_fraction * n))
            val_end = train_end + int(round(val_fraction * n))
            cal_end = val_end + int(round(cal_fraction * n))

            # Garantir au moins une ligne quand c'est possible.
            train_end = max(1, min(train_end, n))
            val_end = max(train_end + 1, min(val_end, n)) if n >= 2 else train_end
            cal_end = max(val_end + 1, min(cal_end, n)) if n >= 3 else val_end

            buckets["train"].append(group.iloc[:train_end])
            buckets["val"].append(group.iloc[train_end:val_end])
            buckets["cal"].append(group.iloc[val_end:cal_end])
            buckets["test"].append(group.iloc[cal_end:])

        splits = {
            name: pd.concat(parts, ignore_index=True).reset_index(drop=True)
            for name, parts in buckets.items()
        }
        return splits

    raise ValueError(f"Protocole inconnu : {protocol}")

def capped_indices_per_class(y: np.ndarray, cap: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected = []
    for cls in np.unique(y):
        indices = np.flatnonzero(y == cls)
        if len(indices) > cap:
            indices = rng.choice(indices, size=cap, replace=False)
        selected.append(indices)
    return np.sort(np.concatenate(selected))

def prepare_split_artifacts(
    splits: Dict[str, pd.DataFrame],
    cfg: Config,
    label_col: str,
) -> Dict[str, object]:
    label_encoder = LabelEncoder()
    label_encoder.fit(splits["train"][label_col])
    class_names = label_encoder.classes_.tolist()
    n_classes = len(class_names)

    all_feature_columns = [
        c for c in splits["train"].columns
        if c not in {label_col, "__source_file__", "__row_in_file__"}
    ]

    X_train_all = splits["train"][all_feature_columns].to_numpy(dtype=np.float64)
    y_train_all = label_encoder.transform(splits["train"][label_col])

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_all)

    variance_filter = VarianceThreshold(0.0)
    X_train_var = variance_filter.fit_transform(X_train_imp)

    variance_features = [
        feature
        for feature, keep in zip(all_feature_columns, variance_filter.get_support())
        if keep
    ]

    selection_indices = capped_indices_per_class(
        y_train_all,
        cfg.feature_selection_rows_per_class,
        cfg.seeds[0],
    )

    selector = RandomForestClassifier(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=cfg.seeds[0],
    )
    selector.fit(X_train_var[selection_indices], y_train_all[selection_indices])

    importance_df = pd.DataFrame({
        "feature": variance_features,
        "importance": selector.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    numeric_features = importance_df.head(min(cfg.numeric_top_k, len(importance_df)))["feature"].tolist()

    # Éviter quelques redondances ou doublons directs.
    cleaned_text_features = []
    seen_prefixes = set()
    for feature in numeric_features:
        prefix = feature.replace("1", "")
        if prefix in seen_prefixes:
            continue
        seen_prefixes.add(prefix)
        cleaned_text_features.append(feature)
        if len(cleaned_text_features) >= cfg.text_top_k:
            break
    text_features = cleaned_text_features

    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    scaler = StandardScaler()

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[all_feature_columns].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        if split_name == "train":
            X_scaled = scaler.fit_transform(X_num).astype(np.float32)
        else:
            X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[label_col]).astype(np.int64),
        }

    return {
        "label_encoder": label_encoder,
        "class_names": class_names,
        "n_classes": n_classes,
        "all_feature_columns": all_feature_columns,
        "numeric_features": numeric_features,
        "text_features": text_features,
        "importance_df": importance_df,
        "imputer": imputer,
        "variance_filter": variance_filter,
        "scaler": scaler,
        "arrays": arrays,
    }

def compact_prompt(text_features: List[str], values: np.ndarray) -> str:
    return "classify: " + " ".join(
        f"{name}={float(value):.4f}"
        for name, value in zip(text_features, values)
    )

class IndexedDataset(Dataset):
    def __init__(self, labels: np.ndarray, indices: np.ndarray):
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item: int):
        idx = int(self.indices[item])
        return {"index": idx, "label": int(self.labels[idx])}

class CompactCollator:
    def __init__(self, tokenizer, scaled_text: np.ndarray, text_features: List[str], max_length: int):
        self.tokenizer = tokenizer
        self.scaled_text = scaled_text
        self.text_features = text_features
        self.max_length = max_length

    def __call__(self, items):
        indices = [item["index"] for item in items]
        labels = torch.tensor([item["label"] for item in items], dtype=torch.long)
        texts = [compact_prompt(self.text_features, self.scaled_text[idx]) for idx in indices]
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        encoded["labels"] = labels
        encoded["indices"] = torch.tensor(indices, dtype=torch.long)
        return encoded



def metrics_from_probs(probabilities: np.ndarray, labels: np.ndarray, n_classes: int) -> Dict[str, float]:
    predictions = probabilities.argmax(axis=1)
    all_labels = np.arange(n_classes)
    present_labels = np.unique(labels)

    macro_f1_present = float(f1_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_f1_all = float(f1_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_precision_present = float(precision_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_precision_all = float(precision_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_recall_present = float(recall_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_recall_all = float(recall_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": macro_f1_present,
        "macro_f1_present": macro_f1_present,
        "macro_f1_all": macro_f1_all,
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "macro_precision": macro_precision_present,
        "macro_precision_present": macro_precision_present,
        "macro_precision_all": macro_precision_all,
        "macro_recall": macro_recall_present,
        "macro_recall_present": macro_recall_present,
        "macro_recall_all": macro_recall_all,
        "nll": float(log_loss(labels, probabilities, labels=all_labels)),
        "present_class_count": int(len(present_labels)),
        "label_space_coverage": float(len(present_labels) / max(1, n_classes)),
    }


def dir_size_mb(path: Path) -> float:
    return float(sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024 ** 2))


def build_class_weights(y_train_small: np.ndarray, n_classes: int, cfg: Config) -> torch.Tensor:
    counts = np.bincount(y_train_small, minlength=n_classes).astype(np.float64)
    if cfg.use_sqrt_class_weights:
        weights = np.sqrt(
            len(y_train_small) / (n_classes * np.maximum(counts, 1.0))
        )
        weights = np.clip(weights, cfg.class_weight_min, cfg.class_weight_max)
        weights = weights / weights.mean()
    else:
        weights = np.ones(n_classes, dtype=np.float64)
    return torch.tensor(weights, dtype=torch.float32)

def build_slm_model(model_checkpoint: str, n_classes: int, class_names: List[str], cfg: Config):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    linear_leaf_names = sorted({
        name.split(".")[-1]
        for name, module in base_model.named_modules()
        if isinstance(module, nn.Linear)
    })
    target_modules = list(cfg.lora_target_modules)
    missing = [name for name in target_modules if name not in linear_leaf_names]
    if missing:
        raise ValueError(f"Modules LoRA introuvables : {missing}")

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=target_modules,
        modules_to_save=list(BACKBONE_SPECS[cfg.current_backbone]["modules_to_save"]),
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    n_total = sum(parameter.numel() for parameter in model.parameters())
    n_trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    n_lora = sum(
        parameter.numel()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and "lora_" in name
    )
    n_head = n_trainable - n_lora

    stats = {
        "llm_total_parameters": n_total,
        "llm_lora_parameters": n_lora,
        "llm_head_parameters": n_head,
        "llm_trainable_parameters": n_trainable,
        "llm_trainable_percent": 100.0 * n_trainable / n_total,
    }
    return model, stats

@torch.no_grad()
def collect_llm_probs(
    model,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, leave=False):
        labels = batch.pop("labels")
        indices = batch.pop("indices")
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        all_indices.append(indices.numpy())
    return (
        np.concatenate(all_probs),
        np.concatenate(all_labels),
        np.concatenate(all_indices),
    )

def train_slm(
    arrays: Dict[str, Dict[str, np.ndarray]],
    text_features: List[str],
    class_names: List[str],
    n_classes: int,
    seed: int,
    run_dir: Path,
    cfg: Config,
) -> Dict[str, object]:
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_checkpoint)

    train_indices = capped_indices_per_class(arrays["train"]["y"], cfg.llm_train_cap_per_class, seed + 10)
    val_indices = capped_indices_per_class(arrays["val"]["y"], cfg.llm_val_cap_per_class, seed + 11)

    class_weights = build_class_weights(arrays["train"]["y"][train_indices], n_classes, cfg)

    train_dataset = IndexedDataset(arrays["train"]["y"], train_indices)
    val_dataset = IndexedDataset(arrays["val"]["y"], val_indices)

    train_collator = CompactCollator(tokenizer, arrays["train"]["scaled_text"], text_features, cfg.max_length)
    val_collator = CompactCollator(tokenizer, arrays["val"]["scaled_text"], text_features, cfg.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_train,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=train_collator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=val_collator,
    )

    model, param_stats = build_slm_model(cfg.model_checkpoint, n_classes, class_names, cfg)
    param_stats["backbone"] = cfg.current_backbone
    param_stats["backbone_display"] = backbone_display_name(cfg.current_backbone)
    param_stats["model_checkpoint"] = cfg.model_checkpoint

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=cfg.label_smoothing,
    )
    optimizer = AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum)
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_fraction * total_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
    amp_enabled = DEVICE.type == "cuda"
    grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_state = None
    best_epoch = -1
    best_macro = -np.inf
    patience = 0
    history = []

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    train_start = time.perf_counter()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"{backbone_display_name(cfg.current_backbone)} seed={seed} epoch {epoch}/{cfg.epochs}",
        )
        for step, batch in enumerate(progress, start=1):
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch.pop("indices")
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(**batch).logits
                loss = criterion(logits, labels) / cfg.grad_accum

            grad_scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * cfg.grad_accum

            if step % cfg.grad_accum == 0 or step == len(train_loader):
                grad_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters() if parameter.requires_grad],
                    max_norm=1.0,
                )
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            progress.set_postfix(loss=f"{running_loss/step:.4f}")

        val_probs_small, val_labels_small, _ = collect_llm_probs(model, val_loader, DEVICE)
        val_metrics_small = metrics_from_probs(val_probs_small, val_labels_small, n_classes)
        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(1, len(train_loader)),
            **{f"val_bal_{key}": value for key, value in val_metrics_small.items()},
        }
        history.append(row)
        print(pd.Series(row).round(6))

        if val_metrics_small["macro_f1"] > best_macro:
            best_macro = val_metrics_small["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience = 0
            print(
                f"🏆 Nouveau meilleur {backbone_display_name(cfg.current_backbone)} "
                f"(seed={seed}) : Macro-F1={best_macro:.6f}"
            )
        else:
            patience += 1
            if patience >= cfg.early_stopping_patience:
                print(f"Early stopping {backbone_display_name(cfg.current_backbone)}.")
                break

    if best_state is None:
        raise RuntimeError("Aucun checkpoint MiniLM sauvegardé.")

    model.load_state_dict(best_state, strict=True)

    llm_train_seconds = time.perf_counter() - train_start
    peak_vram_gib = (
        torch.cuda.max_memory_allocated() / 2**30
        if DEVICE.type == "cuda"
        else 0.0
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "minilm_training_history.csv", index=False)

    return {
        "tokenizer": tokenizer,
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_balanced_val_macro_f1": best_macro,
        "train_seconds": llm_train_seconds,
        "peak_vram_gib": peak_vram_gib,
        "param_stats": param_stats,
        "train_indices": train_indices,
        "val_indices": val_indices,
        "class_weights": class_weights.numpy(),
    }

def train_xgboost(
    arrays: Dict[str, Dict[str, np.ndarray]],
    n_classes: int,
    seed: int,
    cfg: Config,
) -> Tuple[object, float]:
    xgb_common = dict(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=cfg.xgb_n_estimators,
        max_depth=cfg.xgb_max_depth,
        learning_rate=cfg.xgb_learning_rate,
        subsample=cfg.xgb_subsample,
        colsample_bytree=cfg.xgb_colsample_bytree,
        min_child_weight=cfg.xgb_min_child_weight,
        reg_lambda=cfg.xgb_reg_lambda,
        max_bin=256,
        tree_method="hist",
        eval_metric="mlogloss",
        early_stopping_rounds=cfg.xgb_early_stopping_rounds,
        random_state=seed,
        n_jobs=-1,
    )

    model = XGBClassifier(
        **xgb_common,
        device="cuda" if DEVICE.type == "cuda" else "cpu",
    )

    start = time.perf_counter()
    try:
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    except Exception as exc:
        print("⚠️ XGBoost GPU indisponible, reprise CPU :", exc)
        model = XGBClassifier(**xgb_common, device="cpu")
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    seconds = time.perf_counter() - start
    return model, seconds

def selection_score(metrics: Dict[str, float], cfg: Config) -> float:
    if cfg.fusion_selection_metric not in metrics:
        raise KeyError(cfg.fusion_selection_metric)
    return metrics[cfg.fusion_selection_metric]

def make_full_llm_loader(
    arrays: Dict[str, Dict[str, np.ndarray]],
    split_name: str,
    tokenizer,
    text_features: List[str],
    cfg: Config,
) -> DataLoader:
    indices = np.arange(len(arrays[split_name]["y"]))
    dataset = IndexedDataset(arrays[split_name]["y"], indices)
    collator = CompactCollator(tokenizer, arrays[split_name]["scaled_text"], text_features, cfg.max_length)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )

def align_probabilities(
    probabilities: np.ndarray,
    source_classes: np.ndarray,
    n_classes: int,
) -> np.ndarray:
    """Aligne les colonnes d'un classifieur entraîné sur un sous-ensemble de classes."""
    probabilities = np.asarray(probabilities, dtype=np.float64)
    source_classes = np.asarray(source_classes, dtype=np.int64)
    aligned = np.zeros((len(probabilities), n_classes), dtype=np.float64)
    aligned[:, source_classes] = probabilities
    row_sums = aligned.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0
    if np.any(zero_rows):
        aligned[zero_rows] = 1.0 / n_classes
        row_sums = aligned.sum(axis=1, keepdims=True)
    return aligned / row_sums


def split_selection_and_conformal_indices(labels: np.ndarray, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    indices = np.arange(len(labels))
    unique, counts = np.unique(labels, return_counts=True)
    stratify = labels if len(unique) > 1 and counts.min() >= 2 else None
    select_idx, conformal_idx = train_test_split(
        indices,
        test_size=0.50,
        random_state=seed + 31,
        stratify=stratify,
    )
    return np.sort(select_idx), np.sort(conformal_idx)


## 5 — Fonction principale d’expérience

In [ ]:
def run_single_experiment(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    cfg: Config,
    output_dir: Path,
) -> Dict[str, object]:
    set_seed(seed)

    backbone = cfg.current_backbone
    backbone_name = backbone_display_name(backbone)
    run_name = f"{protocol}__{backbone}__seed_{seed}"
    run_dir = output_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"RUN : {run_name} | {backbone_name}")
    print("=" * 90)

    splits = make_splits(
        data=data,
        protocol=protocol,
        seed=seed,
        label_col=LABEL_COL,
        train_fraction=cfg.train_fraction,
        val_fraction=cfg.val_fraction,
        cal_fraction=cfg.cal_fraction,
        test_fraction=cfg.test_fraction,
    )

    split_sizes = []
    for split_name, split_df in splits.items():
        counts = split_df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
        counts["split"] = split_name
        counts["protocol"] = protocol
        counts["backbone"] = backbone
        counts["seed"] = seed
        split_sizes.append(counts)
        print(f"\n{split_name.upper():5s}: {len(split_df):,} lignes")
        print(split_df[LABEL_COL].value_counts().to_string())
    split_distribution_df = pd.concat(split_sizes, ignore_index=True)
    split_distribution_df.to_csv(run_dir / "split_distribution.csv", index=False)

    artifacts = prepare_split_artifacts(splits, cfg, LABEL_COL)
    arrays = artifacts["arrays"]
    class_names = artifacts["class_names"]
    n_classes = artifacts["n_classes"]
    numeric_features = artifacts["numeric_features"]
    text_features = artifacts["text_features"]
    importance_df = artifacts["importance_df"]

    importance_df.to_csv(run_dir / "feature_selection_importance.csv", index=False)

    # Audit de longueur des tokens sur un sous-échantillon train.
    audit_size = min(10_000, len(arrays["train"]["y"]))
    audit_indices = np.arange(audit_size)
    tokenizer_audit = AutoTokenizer.from_pretrained(cfg.model_checkpoint)
    token_lengths = []
    for idx in tqdm(audit_indices, desc=f"Audit tokens {run_name}"):
        prompt = compact_prompt(text_features, arrays["train"]["scaled_text"][idx])
        token_lengths.append(len(tokenizer_audit(prompt, truncation=False, add_special_tokens=True)["input_ids"]))
    token_lengths = np.asarray(token_lengths, dtype=np.int64)
    token_audit = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "n_audit": int(audit_size),
        "text_feature_count": int(len(text_features)),
        "max_length": int(cfg.max_length),
        "mean_tokens": float(token_lengths.mean()),
        "median_tokens": float(np.median(token_lengths)),
        "p95_tokens": float(np.percentile(token_lengths, 95)),
        "max_tokens": int(token_lengths.max()),
        "truncation_rate": float(np.mean(token_lengths > cfg.max_length)),
    }
    pd.DataFrame([token_audit]).to_csv(run_dir / "token_audit.csv", index=False)

    # MiniLM
    llm_bundle = train_slm(arrays, text_features, class_names, n_classes, seed, run_dir, cfg)
    llm_model = llm_bundle["model"]
    llm_tokenizer = llm_bundle["tokenizer"]

    # XGBoost
    xgb_model, xgb_train_seconds = train_xgboost(arrays, n_classes, seed, cfg)

    probabilities = {}
    for split_name in ["val", "cal", "test"]:
        llm_loader = make_full_llm_loader(arrays, split_name, llm_tokenizer, text_features, cfg)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        llm_probs, labels, indices = collect_llm_probs(llm_model, llm_loader, DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        llm_seconds = time.perf_counter() - start

        xgb_probs = xgb_model.predict_proba(arrays[split_name]["raw_numeric"])
        xgb_probs = np.asarray(xgb_probs, dtype=np.float64)
        xgb_probs = xgb_probs / xgb_probs.sum(axis=1, keepdims=True)

        probabilities[split_name] = {
            "llm": llm_probs.astype(np.float64),
            "xgb": xgb_probs,
            "labels": labels,
            "indices": indices,
            "llm_seconds": llm_seconds,
        }

    # CAL est séparé : une moitié pour sélectionner la fusion, l'autre pour la conformité.
    cal_labels_all = probabilities["cal"]["labels"]
    cal_llm_all = probabilities["cal"]["llm"]
    cal_xgb_all = probabilities["cal"]["xgb"]
    cal_select_idx, cal_conformal_idx = split_selection_and_conformal_indices(cal_labels_all, seed)

    cal_labels = cal_labels_all[cal_select_idx]
    cal_llm = cal_llm_all[cal_select_idx]
    cal_xgb = cal_xgb_all[cal_select_idx]

    np.savez(
        run_dir / "calibration_partition_indices.npz",
        selection_indices=cal_select_idx,
        conformal_indices=cal_conformal_idx,
    )

    candidate_cal_metrics: Dict[str, Dict[str, float]] = {}
    candidate_test_probs: Dict[str, np.ndarray] = {}

    candidate_cal_metrics["slm"] = metrics_from_probs(cal_llm, cal_labels, n_classes)
    candidate_test_probs["slm"] = probabilities["test"]["llm"]

    candidate_cal_metrics["xgboost"] = metrics_from_probs(cal_xgb, cal_labels, n_classes)
    candidate_test_probs["xgboost"] = probabilities["test"]["xgb"]

    minimum_alpha = 0.0
    maximum_alpha = 1.0 - cfg.fusion_min_semantic_weight
    alphas = np.linspace(minimum_alpha, maximum_alpha, cfg.fusion_grid_points)

    best_alpha = None
    best_alpha_metrics = None
    best_alpha_score = -np.inf

    for alpha in alphas:
        fused = alpha * cal_xgb + (1.0 - alpha) * cal_llm
        fused = fused / fused.sum(axis=1, keepdims=True)
        metrics = metrics_from_probs(fused, cal_labels, n_classes)
        score = selection_score(metrics, cfg)
        if score > best_alpha_score:
            best_alpha_score = score
            best_alpha = float(alpha)
            best_alpha_metrics = metrics

    candidate_name_fusion = f"fusion_alpha_{best_alpha:.3f}"
    candidate_cal_metrics[candidate_name_fusion] = best_alpha_metrics

    test_alpha_probs = (
        best_alpha * probabilities["test"]["xgb"]
        + (1.0 - best_alpha) * probabilities["test"]["llm"]
    )
    test_alpha_probs /= test_alpha_probs.sum(axis=1, keepdims=True)
    candidate_test_probs[candidate_name_fusion] = test_alpha_probs

    eps = 1e-8
    stack_X_cal = np.concatenate(
        [np.log(cal_xgb + eps), np.log(cal_llm + eps)],
        axis=1,
    )
    stacker = LogisticRegression(
        C=10.0,
        max_iter=1000,
        solver="lbfgs",
        multi_class="multinomial",
        n_jobs=-1,
        random_state=seed,
    )
    stacker.fit(stack_X_cal, cal_labels)
    stack_cal_probs = align_probabilities(
        stacker.predict_proba(stack_X_cal),
        stacker.classes_,
        n_classes,
    )
    candidate_cal_metrics["stacking"] = metrics_from_probs(stack_cal_probs, cal_labels, n_classes)

    stack_X_test = np.concatenate(
        [
            np.log(probabilities["test"]["xgb"] + eps),
            np.log(probabilities["test"]["llm"] + eps),
        ],
        axis=1,
    )
    candidate_test_probs["stacking"] = align_probabilities(
        stacker.predict_proba(stack_X_test),
        stacker.classes_,
        n_classes,
    )

    cal_comparison = pd.DataFrame(candidate_cal_metrics).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    cal_comparison.to_csv(run_dir / "candidate_calibration_comparison.csv")

    selected_candidate = cal_comparison.index[0]
    final_probs = candidate_test_probs[selected_candidate]
    test_labels = probabilities["test"]["labels"]
    final_predictions = final_probs.argmax(axis=1)
    final_metrics = metrics_from_probs(final_probs, test_labels, n_classes)
    final_report = classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    final_cm = confusion_matrix(test_labels, final_predictions, labels=np.arange(n_classes))

    test_results = {
        name: metrics_from_probs(probs, test_labels, n_classes)
        for name, probs in candidate_test_probs.items()
    }
    test_comparison = pd.DataFrame(test_results).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    test_comparison.to_csv(run_dir / "candidate_test_comparison.csv")

    branch_summary = pd.DataFrame({
        f"{backbone_name}-LoRA": test_results["slm"],
        "XGBoost": test_results["xgboost"],
        "Final": test_results[selected_candidate],
    }).T
    branch_summary["delta_weighted_f1_vs_minilm"] = (
        branch_summary["weighted_f1"] - branch_summary.loc[f"{backbone_name}-LoRA", "weighted_f1"]
    )
    branch_summary["delta_macro_f1_vs_minilm"] = (
        branch_summary["macro_f1"] - branch_summary.loc[f"{backbone_name}-LoRA", "macro_f1"]
    )
    if "macro_f1_all" in branch_summary.columns:
        branch_summary["delta_macro_f1_all_vs_minilm"] = (
            branch_summary["macro_f1_all"] - branch_summary.loc[f"{backbone_name}-LoRA", "macro_f1_all"]
        )
    branch_summary.to_csv(run_dir / "branch_summary.csv")

    # Sauvegardes des modèles et prédictions.
    adapter_dir = run_dir / "slm_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    llm_model.save_pretrained(adapter_dir)
    llm_tokenizer.save_pretrained(adapter_dir)
    adapter_size_mb = dir_size_mb(adapter_dir)

    joblib.dump(xgb_model, run_dir / "xgboost_model.joblib")
    joblib.dump(stacker, run_dir / "fusion_stacker.joblib")
    joblib.dump(artifacts["label_encoder"], run_dir / "label_encoder.joblib")
    joblib.dump(artifacts["imputer"], run_dir / "imputer.joblib")
    joblib.dump(artifacts["variance_filter"], run_dir / "variance_filter.joblib")
    joblib.dump(artifacts["scaler"], run_dir / "scaler.joblib")
    joblib.dump(artifacts["all_feature_columns"], run_dir / "input_features.joblib")
    joblib.dump(numeric_features, run_dir / "numeric_features.joblib")
    joblib.dump(text_features, run_dir / "text_features.joblib")

    prediction_df = pd.DataFrame({
        "true_label": [class_names[index] for index in test_labels],
        "predicted_label": [class_names[index] for index in final_predictions],
        "confidence": final_probs.max(axis=1),
    })
    for class_index, class_name in enumerate(class_names):
        prediction_df[f"p_{class_name}"] = final_probs[:, class_index]
    prediction_df.to_csv(run_dir / "test_predictions.csv", index=False)

    with open(run_dir / "classification_report.json", "w", encoding="utf-8") as handle:
        json.dump(final_report, handle, indent=2, ensure_ascii=False)
    np.save(run_dir / "confusion_matrix.npy", final_cm)

    metadata_dict = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "lora_target_modules": list(cfg.lora_target_modules),
        "calibration_selection_size": int(len(cal_select_idx)),
        "conformal_calibration_size": int(len(cal_conformal_idx)),
        "text_features": text_features,
        "numeric_features": numeric_features,
        "token_audit": token_audit,
        "param_stats": llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
    }
    with open(run_dir / "run_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(metadata_dict, handle, indent=2, ensure_ascii=False)

    # Libération mémoire.
    del llm_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    summary_row = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "lora_target_modules": list(cfg.lora_target_modules),
        "n_classes": n_classes,
        "classes": " | ".join(class_names),
        "numeric_features": " | ".join(numeric_features),
        "text_features": " | ".join(text_features),
        **llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
        **token_audit,
        **{f"final_{key}": value for key, value in final_metrics.items()},
        **{f"slm_test_{key}": value for key, value in test_results["slm"].items()},
        **{f"xgboost_test_{key}": value for key, value in test_results["xgboost"].items()},
    }

    print("\nRÉSULTAT FINAL")
    print(pd.Series(summary_row).head(30))
    print("\nRapport final :")
    print(classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
    ))

    return {
        "summary_row": summary_row,
        "split_distribution_df": split_distribution_df,
        "importance_df": importance_df.assign(protocol=protocol, backbone=backbone, seed=seed),
        "token_audit": token_audit,
        "history_df": llm_bundle["history_df"].assign(protocol=protocol, backbone=backbone, seed=seed),
        "branch_summary": branch_summary.assign(protocol=protocol, backbone=backbone, seed=seed),
        "cal_comparison": cal_comparison.assign(protocol=protocol, backbone=backbone, seed=seed),
        "test_comparison": test_comparison.assign(protocol=protocol, backbone=backbone, seed=seed),
        "confusion_matrix": final_cm,
        "class_names": class_names,
        "run_dir": str(run_dir),
        "selected_candidate": selected_candidate,
    }

## 6 — Fonctions de calibration conforme et d’explicabilité v6

In [ ]:

# ================================================================
# TRUSTWORTHINESS v6 : calibration, conformité et explicabilité
# ================================================================

def transform_splits_from_saved_preprocessing(
    splits: Dict[str, pd.DataFrame],
    run_dir: Path,
) -> Dict[str, object]:
    label_encoder = joblib.load(run_dir / "label_encoder.joblib")
    imputer = joblib.load(run_dir / "imputer.joblib")
    variance_filter = joblib.load(run_dir / "variance_filter.joblib")
    scaler = joblib.load(run_dir / "scaler.joblib")
    input_features = joblib.load(run_dir / "input_features.joblib")
    numeric_features = joblib.load(run_dir / "numeric_features.joblib")
    text_features = joblib.load(run_dir / "text_features.joblib")

    variance_features = [
        feature for feature, keep in zip(input_features, variance_filter.get_support()) if keep
    ]
    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[input_features].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[LABEL_COL]).astype(np.int64),
        }

    return {
        "arrays": arrays,
        "label_encoder": label_encoder,
        "class_names": label_encoder.classes_.tolist(),
        "n_classes": len(label_encoder.classes_),
        "numeric_features": numeric_features,
        "text_features": text_features,
    }


def load_slm_adapter(
    run_dir: Path,
    n_classes: int,
    class_names: List[str],
    model_checkpoint: str,
):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    model = PeftModel.from_pretrained(base_model, run_dir / "slm_adapter").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(run_dir / "slm_adapter")
    model.eval()
    return model, tokenizer


def get_candidate_probabilities(
    candidate: str,
    llm_probs: np.ndarray,
    xgb_probs: np.ndarray,
    stacker,
    best_alpha: float,
    n_classes: int,
) -> np.ndarray:
    if candidate == "slm":
        probs = llm_probs
    elif candidate == "xgboost":
        probs = xgb_probs
    elif candidate.startswith("fusion_alpha_"):
        probs = best_alpha * xgb_probs + (1.0 - best_alpha) * llm_probs
    elif candidate == "stacking":
        eps = 1e-8
        features = np.concatenate([np.log(xgb_probs + eps), np.log(llm_probs + eps)], axis=1)
        probs = align_probabilities(stacker.predict_proba(features), stacker.classes_, n_classes)
    else:
        raise ValueError(f"Candidat inconnu : {candidate}")
    probs = np.asarray(probs, dtype=np.float64)
    return probs / probs.sum(axis=1, keepdims=True)


def fit_temperature(probabilities: np.ndarray, labels: np.ndarray) -> float:
    logits = torch.tensor(np.log(np.clip(probabilities, 1e-12, 1.0)), dtype=torch.float64)
    targets = torch.tensor(labels, dtype=torch.long)
    log_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=80, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature).clamp(0.05, 20.0)
        loss = F.cross_entropy(logits / temperature, targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(log_temperature.detach()).clamp(0.05, 20.0).item())


def apply_temperature(probabilities: np.ndarray, temperature: float) -> np.ndarray:
    logits = np.log(np.clip(probabilities, 1e-12, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp_values = np.exp(logits)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def conformal_true_scores(
    probabilities: np.ndarray,
    labels: np.ndarray,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> np.ndarray:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(len(order))[:, None], order] = np.arange(probabilities.shape[1])[None, :]
    true_rank = ranks[np.arange(len(labels)), labels]
    scores = cumulative[np.arange(len(labels)), true_rank]
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(true_rank + 1 - k_reg, 0)
    return scores


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = min(1.0, math.ceil((n + 1) * (1.0 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def build_prediction_sets(
    probabilities: np.ndarray,
    qhat: float,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> List[List[int]]:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    rank_numbers = np.arange(1, probabilities.shape[1] + 1)
    scores = cumulative.copy()
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(rank_numbers - k_reg, 0)[None, :]

    sets = []
    for i in range(len(probabilities)):
        include = scores[i] <= qhat
        include[0] = True
        sets.append(order[i][include].tolist())
    return sets


def evaluate_prediction_sets(
    prediction_sets: List[List[int]],
    labels: np.ndarray,
    class_names: List[str],
    abstain_size: int,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    sizes = np.asarray([len(values) for values in prediction_sets], dtype=np.int64)
    covered = np.asarray([int(label in values) for label, values in zip(labels, prediction_sets)])
    overall = {
        "coverage": float(covered.mean()),
        "average_set_size": float(sizes.mean()),
        "median_set_size": float(np.median(sizes)),
        "singleton_rate": float(np.mean(sizes == 1)),
        "ambiguity_rate": float(np.mean((sizes > 1) & (sizes <= abstain_size))),
        "abstention_rate": float(np.mean(sizes > abstain_size)),
    }
    rows = []
    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        rows.append({
            "class": class_name,
            "support": int(mask.sum()),
            "coverage": float(covered[mask].mean()) if mask.any() else np.nan,
            "average_set_size": float(sizes[mask].mean()) if mask.any() else np.nan,
        })
    return overall, pd.DataFrame(rows)


def stratified_explanation_indices(labels: np.ndarray, n_samples: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    present_classes = np.unique(labels)
    per_class = max(1, n_samples // max(1, len(present_classes)))
    selected = []
    for class_index in present_classes:
        indices = np.flatnonzero(labels == class_index)
        take = min(per_class, len(indices))
        selected.extend(rng.choice(indices, size=take, replace=False).tolist())
    remaining = n_samples - len(selected)
    if remaining > 0:
        pool = np.setdiff1d(np.arange(len(labels)), np.asarray(selected, dtype=np.int64))
        if len(pool):
            selected.extend(rng.choice(pool, size=min(remaining, len(pool)), replace=False).tolist())
    return np.asarray(sorted(selected[:n_samples]), dtype=np.int64)


def get_tree_shap_values(xgb_model, X: np.ndarray, n_classes: int) -> np.ndarray:
    raw = xgb_model.get_booster().predict(
        xgb.DMatrix(X),
        pred_contribs=True,
        strict_shape=True,
    )
    raw = np.asarray(raw)
    if raw.ndim == 3:
        return raw[:, :, :-1]
    if raw.ndim == 2 and n_classes > 1:
        return raw.reshape(len(X), n_classes, -1)[:, :, :-1]
    raise RuntimeError(f"Forme TreeSHAP inattendue : {raw.shape}")


def slm_predict_values(model, tokenizer, text_features: List[str], values_batch: np.ndarray) -> np.ndarray:
    prompts = [compact_prompt(text_features, values) for values in values_batch]
    encoded = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=CFG.max_length,
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()


def deterministic_template_explanation(
    predicted_label: str,
    confidence: float,
    conformal_labels: List[str],
    shap_evidence: List[Tuple[str, float]],
    occlusion_evidence: List[Tuple[str, float]],
) -> str:
    numeric_text = ", ".join(f"[{name}] ({value:+.3f})" for name, value in shap_evidence)
    semantic_text = ", ".join(f"[{name}] (drop={value:.3f})" for name, value in occlusion_evidence)
    uncertainty = (
        f"The conformal set contains only {predicted_label}."
        if len(conformal_labels) == 1
        else f"The decision is uncertain among: {', '.join(conformal_labels)}."
    )
    return (
        f"The predicted class is {predicted_label} with confidence {confidence:.3f}. "
        f"The strongest numeric evidence is {numeric_text}. "
        f"The SLM evidence is {semantic_text}. {uncertainty}"
    )


_EXPLAINER_CACHE = {}

def get_explanation_slm():
    checkpoint = CFG.explanation_model_checkpoint
    if checkpoint not in _EXPLAINER_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(DEVICE)
        model.eval()
        _EXPLAINER_CACHE[checkpoint] = (model, tokenizer)
    return _EXPLAINER_CACHE[checkpoint]


def generate_slm_explanations(prompts: List[str]) -> Tuple[List[str], float]:
    model, tokenizer = get_explanation_slm()
    start = time.perf_counter()
    outputs = []
    batch_size = 8
    for start_index in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start_index:start_index + batch_size]
        encoded = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=CFG.explanation_max_new_tokens,
                do_sample=False,
                num_beams=2,
                repetition_penalty=1.1,
            )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    seconds = time.perf_counter() - start
    return outputs, seconds


def citation_metrics(text: str, allowed_features: List[str]) -> Dict[str, float]:
    citations = re.findall(r"\[([a-zA-Z0-9_]+)\]", text)
    allowed = set(allowed_features)
    valid = [citation for citation in citations if citation in allowed]
    unsupported = [citation for citation in citations if citation not in allowed]
    return {
        "citation_count": len(citations),
        "valid_citation_rate": len(valid) / max(1, len(citations)),
        "unsupported_citation_rate": len(unsupported) / max(1, len(citations)),
        "evidence_coverage": len(set(valid)) / max(1, len(allowed)),
    }


## 6 — Reprise automatique des runs sauvegardés

La cellule suivante vérifie les 12 dossiers attendus dans Google Drive.

Un run est considéré comme terminé uniquement si ses modèles, métadonnées,
prédictions, métriques et matrice de confusion sont tous présents.

- `resume_completed_runs=True` : recharge les runs terminés.
- `force_rerun=False` : interdit leur réentraînement.
- Un run interrompu avant sa sauvegarde finale est le seul run relancé.

In [ ]:
def expected_run_dir(
    output_dir: Path,
    protocol: str,
    backbone: str,
    seed: int,
) -> Path:
    return output_dir / f"{protocol}__{backbone}__seed_{seed}"


def completed_run_files(run_dir: Path) -> List[Path]:
    return [
        run_dir / "run_metadata.json",
        run_dir / "candidate_test_comparison.csv",
        run_dir / "candidate_calibration_comparison.csv",
        run_dir / "branch_summary.csv",
        run_dir / "feature_selection_importance.csv",
        run_dir / "split_distribution.csv",
        run_dir / "token_audit.csv",
        run_dir / "minilm_training_history.csv",
        run_dir / "confusion_matrix.npy",
        run_dir / "classification_report.json",
        run_dir / "label_encoder.joblib",
        run_dir / "test_predictions.csv",
        run_dir / "slm_adapter",
        run_dir / "xgboost_model.joblib",
        run_dir / "fusion_stacker.joblib",
    ]


def is_run_complete(run_dir: Path) -> bool:
    if not run_dir.exists():
        return False

    for path in completed_run_files(run_dir):
        if not path.exists():
            return False
        if path.is_file() and path.stat().st_size == 0:
            return False

    try:
        with open(
            run_dir / "run_metadata.json",
            "r",
            encoding="utf-8",
        ) as handle:
            metadata_dict = json.load(handle)

        selected = metadata_dict["selected_candidate"]
        test_table = pd.read_csv(
            run_dir / "candidate_test_comparison.csv",
            index_col=0,
        )
        if selected not in test_table.index:
            return False

        predictions = pd.read_csv(
            run_dir / "test_predictions.csv",
            nrows=5,
        )
        if predictions.empty:
            return False

    except Exception as exc:
        print(f"Run incomplet ou illisible {run_dir.name}: {exc}")
        return False

    return True


def load_completed_run(run_dir: Path) -> Dict[str, object]:
    """Reconstruit le résultat d'un run sans recharger ses modèles."""
    with open(
        run_dir / "run_metadata.json",
        "r",
        encoding="utf-8",
    ) as handle:
        metadata_dict = json.load(handle)

    protocol = str(metadata_dict["protocol"])
    backbone = str(metadata_dict["backbone"])
    backbone_name = str(metadata_dict["backbone_display"])
    seed = int(metadata_dict["seed"])
    selected_candidate = str(metadata_dict["selected_candidate"])

    label_encoder = joblib.load(run_dir / "label_encoder.joblib")
    class_names = label_encoder.classes_.tolist()
    n_classes = len(class_names)

    test_comparison = pd.read_csv(
        run_dir / "candidate_test_comparison.csv",
        index_col=0,
    )
    cal_comparison = pd.read_csv(
        run_dir / "candidate_calibration_comparison.csv",
        index_col=0,
    )
    branch_summary = pd.read_csv(
        run_dir / "branch_summary.csv",
        index_col=0,
    )
    importance_df = pd.read_csv(
        run_dir / "feature_selection_importance.csv",
    )
    split_distribution_df = pd.read_csv(
        run_dir / "split_distribution.csv",
    )
    history_df = pd.read_csv(
        run_dir / "minilm_training_history.csv",
    )
    token_audit_df = pd.read_csv(
        run_dir / "token_audit.csv",
    )

    if selected_candidate not in test_comparison.index:
        raise KeyError(
            f"Candidat {selected_candidate} absent de "
            f"{run_dir / 'candidate_test_comparison.csv'}"
        )

    final_metrics = test_comparison.loc[selected_candidate].to_dict()
    slm_metrics = test_comparison.loc["slm"].to_dict()
    xgb_metrics = test_comparison.loc["xgboost"].to_dict()

    param_stats = dict(metadata_dict.get("param_stats", {}))
    token_audit = dict(metadata_dict.get("token_audit", {}))

    summary_row = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": metadata_dict["model_checkpoint"],
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": float(
            metadata_dict["best_alpha_numeric"]
        ),
        "lora_target_modules": metadata_dict.get(
            "lora_target_modules",
            BACKBONE_SPECS[backbone]["target_modules"],
        ),
        "n_classes": n_classes,
        "classes": " | ".join(class_names),
        "numeric_features": " | ".join(
            metadata_dict["numeric_features"]
        ),
        "text_features": " | ".join(
            metadata_dict["text_features"]
        ),
        **param_stats,
        "llm_train_seconds": float(
            metadata_dict["llm_train_seconds"]
        ),
        "xgb_train_seconds": float(
            metadata_dict["xgb_train_seconds"]
        ),
        "peak_vram_gib": float(
            metadata_dict["peak_vram_gib"]
        ),
        "adapter_size_mb": float(
            metadata_dict.get(
                "adapter_size_mb",
                dir_size_mb(run_dir / "slm_adapter"),
            )
        ),
        **token_audit,
        **{
            f"final_{key}": value
            for key, value in final_metrics.items()
        },
        **{
            f"slm_test_{key}": value
            for key, value in slm_metrics.items()
        },
        **{
            f"xgboost_test_{key}": value
            for key, value in xgb_metrics.items()
        },
    }

    # Ensure context columns exist exactly once.
    for frame in [
        history_df,
        importance_df,
        branch_summary,
        cal_comparison,
        test_comparison,
    ]:
        frame["protocol"] = protocol
        frame["backbone"] = backbone
        frame["seed"] = seed

    if "protocol" not in split_distribution_df.columns:
        split_distribution_df["protocol"] = protocol
    if "backbone" not in split_distribution_df.columns:
        split_distribution_df["backbone"] = backbone
    if "seed" not in split_distribution_df.columns:
        split_distribution_df["seed"] = seed

    if token_audit_df.empty:
        token_audit_record = token_audit
    else:
        token_audit_record = token_audit_df.iloc[0].to_dict()
    token_audit_record["protocol"] = protocol
    token_audit_record["backbone"] = backbone
    token_audit_record["seed"] = seed

    cm = np.load(run_dir / "confusion_matrix.npy")

    print(f"✅ Rechargé sans entraînement : {run_dir.name}")

    return {
        "summary_row": summary_row,
        "split_distribution_df": split_distribution_df,
        "importance_df": importance_df,
        "token_audit": token_audit_record,
        "history_df": history_df,
        "branch_summary": branch_summary,
        "cal_comparison": cal_comparison,
        "test_comparison": test_comparison,
        "confusion_matrix": cm,
        "class_names": class_names,
        "run_dir": str(run_dir),
        "selected_candidate": selected_candidate,
        "loaded_from_disk": True,
    }


def campaign_status_table(
    output_dir: Path,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    for backbone in cfg.backbones:
        for protocol in cfg.protocols:
            for seed in cfg.seeds:
                run_dir = expected_run_dir(
                    output_dir,
                    protocol,
                    backbone,
                    seed,
                )
                rows.append({
                    "backbone": backbone,
                    "protocol": protocol,
                    "seed": seed,
                    "complete": is_run_complete(run_dir),
                    "run_dir": str(run_dir),
                })
    return pd.DataFrame(rows)

## 7 — Chargement strict des 12 runs sauvegardés

Cette cellule ne contient aucun appel d'entraînement. Si un run est incomplet,
elle s'arrête et indique son dossier au lieu de recommencer l'expérience.

In [ ]:
if CFG.allow_detector_training:
    raise RuntimeError(
        "Cette extension doit rester post-hoc. "
        "CFG.allow_detector_training doit être False."
    )

status_before = campaign_status_table(MODEL_DIR, CFG)
display(status_before)

missing_runs = status_before.loc[
    ~status_before["complete"],
    ["protocol", "backbone", "seed", "run_dir"],
]

if not missing_runs.empty:
    display(missing_runs)
    raise RuntimeError(
        "Certains runs sont incomplets. "
        "Cette extension refuse de réentraîner les modèles."
    )

all_summaries = []
all_histories = []
all_importances = []
all_branch_summaries = []
all_split_distributions = []
all_cal_comparisons = []
all_test_comparisons = []
all_token_audits = []
confusion_registry = {}

for backbone in CFG.backbones:
    apply_backbone_config(backbone)
    for protocol in CFG.protocols:
        for seed in CFG.seeds:
            run_dir = expected_run_dir(
                MODEL_DIR,
                protocol,
                backbone,
                seed,
            )
            result = load_completed_run(run_dir)

            all_summaries.append(result["summary_row"])
            all_histories.append(result["history_df"])
            all_importances.append(result["importance_df"])
            all_branch_summaries.append(result["branch_summary"])
            all_split_distributions.append(
                result["split_distribution_df"]
            )
            all_cal_comparisons.append(result["cal_comparison"])
            all_test_comparisons.append(result["test_comparison"])
            all_token_audits.append(result["token_audit"])

            confusion_registry[(protocol, backbone, seed)] = {
                "cm": result["confusion_matrix"],
                "class_names": result["class_names"],
                "selected_candidate": result["selected_candidate"],
                "run_dir": result["run_dir"],
            }

summaries_df = pd.DataFrame(all_summaries)
histories_df = pd.concat(all_histories, ignore_index=True)
feature_importances_df = pd.concat(all_importances, ignore_index=True)
branch_summaries_df = pd.concat(
    all_branch_summaries,
    ignore_index=False,
)
split_distributions_df = pd.concat(
    all_split_distributions,
    ignore_index=True,
)
cal_comparisons_df = pd.concat(
    all_cal_comparisons,
    ignore_index=False,
)
test_comparisons_df = pd.concat(
    all_test_comparisons,
    ignore_index=False,
)
token_audits_df = pd.DataFrame(all_token_audits)

print(f"✅ Runs rechargés sans entraînement : {len(summaries_df)}/12")
display(
    summaries_df[[
        "protocol",
        "backbone_display",
        "seed",
        "selected_candidate",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_weighted_f1",
    ]]
)

## 8 — Configuration de l'extension explicabilité

Le notebook choisit le meilleur run sauvegardé pour chaque
**protocole × backbone**. Les résultats intermédiaires sont sauvegardés dans
`posthoc_explainability_v11/cache/`.

Une exécution interrompue reprend les échantillons manquants sans refaire les
explications déjà calculées.

In [ ]:
best_runs_df = (
    summaries_df
    .sort_values(
        ["protocol", "backbone", "final_weighted_f1"],
        ascending=[True, True, False],
    )
    .groupby(["protocol", "backbone"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_runs_df.to_csv(
    POSTHOC_TABLE_DIR / "selected_runs_for_posthoc_xai.csv",
    index=False,
)

display(
    best_runs_df[[
        "protocol",
        "backbone",
        "backbone_display",
        "seed",
        "selected_candidate",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_weighted_f1",
    ]]
)

print("Sortie post-hoc :", POSTHOC_DIR)

## 9 — Fonctions de reprise, perturbation et validation

In [ ]:
def scenario_name(protocol: str, backbone: str, seed: int) -> str:
    return f"{protocol}__{backbone}__seed_{seed}"


def scenario_cache_dir(
    protocol: str,
    backbone: str,
    seed: int,
) -> Path:
    path = POSTHOC_CACHE_DIR / scenario_name(
        protocol,
        backbone,
        seed,
    )
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_csv_or_empty(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def jaccard_top_k(
    left: List[int],
    right: List[int],
    k: int,
) -> float:
    left_set = set(list(left)[:k])
    right_set = set(list(right)[:k])
    return len(left_set & right_set) / max(
        1,
        len(left_set | right_set),
    )


def safe_spearman(
    left: np.ndarray,
    right: np.ndarray,
) -> float:
    if spearmanr is None:
        return np.nan
    if np.allclose(left, left[0]) or np.allclose(right, right[0]):
        return np.nan
    result = spearmanr(left, right)
    return float(result.correlation)


def select_posthoc_indices(
    labels: np.ndarray,
    predictions: np.ndarray,
    total_samples: int,
    error_quota: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected = []

    present_classes = np.unique(labels)
    per_class = max(
        1,
        (total_samples - error_quota)
        // max(1, len(present_classes)),
    )

    for class_index in present_classes:
        class_indices = np.flatnonzero(labels == class_index)
        take = min(per_class, len(class_indices))
        if take:
            selected.extend(
                rng.choice(
                    class_indices,
                    size=take,
                    replace=False,
                ).tolist()
            )

    error_indices = np.flatnonzero(predictions != labels)
    if len(error_indices):
        take_errors = min(error_quota, len(error_indices))
        selected.extend(
            rng.choice(
                error_indices,
                size=take_errors,
                replace=False,
            ).tolist()
        )

    selected = np.unique(
        np.asarray(selected, dtype=np.int64)
    )

    if len(selected) < total_samples:
        remaining_pool = np.setdiff1d(
            np.arange(len(labels)),
            selected,
        )
        remaining = min(
            total_samples - len(selected),
            len(remaining_pool),
        )
        if remaining:
            selected = np.concatenate([
                selected,
                rng.choice(
                    remaining_pool,
                    size=remaining,
                    replace=False,
                ),
            ])

    return np.sort(selected[:total_samples])


def conformal_category(set_size: int) -> str:
    if set_size == 1:
        return "singleton"
    if set_size <= CFG.conformal_abstain_size:
        return "ambiguous"
    return "abstention"


def build_structured_prompt(
    predicted_label: str,
    confidence: float,
    conformal_labels: List[str],
    numeric_features: List[str],
    semantic_features: List[str],
) -> str:
    return (
        "Return exactly five lines and copy labels/features exactly.\n"
        f"Predicted class: {predicted_label}\n"
        f"Conformal set: {', '.join(conformal_labels)}\n"
        "Numeric evidence: "
        + ", ".join(f"[{name}]" for name in numeric_features)
        + "\nSemantic evidence: "
        + ", ".join(f"[{name}]" for name in semantic_features)
        + "\nAssessment: explain the decision in one short sentence. "
        "Do not introduce any other feature, class, cause, port, "
        "protocol, or attack mechanism."
    )


def validate_structured_explanation(
    text: str,
    predicted_label: str,
    numeric_features: List[str],
    semantic_features: List[str],
) -> Dict[str, object]:
    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]
    citations = re.findall(
        r"\[([a-zA-Z0-9_]+)\]",
        str(text),
    )
    allowed = set(numeric_features + semantic_features)
    unsupported = [
        citation
        for citation in citations
        if citation not in allowed
    ]

    predicted_ok = any(
        line.lower()
        == f"predicted class: {predicted_label}".lower()
        for line in lines
    )

    numeric_line = next(
        (
            line
            for line in lines
            if line.lower().startswith("numeric evidence:")
        ),
        "",
    )
    semantic_line = next(
        (
            line
            for line in lines
            if line.lower().startswith("semantic evidence:")
        ),
        "",
    )

    numeric_ok = any(
        f"[{name}]" in numeric_line
        for name in numeric_features
    )
    semantic_ok = any(
        f"[{name}]" in semantic_line
        for name in semantic_features
    )

    required_prefixes = [
        "predicted class:",
        "conformal set:",
        "numeric evidence:",
        "semantic evidence:",
        "assessment:",
    ]
    structure_ok = (
        len(lines) == 5
        and all(
            any(line.lower().startswith(prefix) for line in lines)
            for prefix in required_prefixes
        )
    )

    valid = bool(
        structure_ok
        and predicted_ok
        and numeric_ok
        and semantic_ok
        and not unsupported
    )

    return {
        "valid": valid,
        "structure_ok": structure_ok,
        "predicted_label_present": predicted_ok,
        "numeric_evidence_present": numeric_ok,
        "semantic_evidence_present": semantic_ok,
        "unsupported_citation_count": len(unsupported),
        "citation_count": len(citations),
    }


def scenario_probability_bundle(
    protocol: str,
    backbone: str,
    seed: int,
) -> Dict[str, object]:
    cache_dir = scenario_cache_dir(
        protocol,
        backbone,
        seed,
    )
    run_dir = expected_run_dir(
        MODEL_DIR,
        protocol,
        backbone,
        seed,
    )

    splits = make_splits(
        data=df,
        protocol=protocol,
        seed=seed,
        label_col=LABEL_COL,
        train_fraction=CFG.train_fraction,
        val_fraction=CFG.val_fraction,
        cal_fraction=CFG.cal_fraction,
        test_fraction=CFG.test_fraction,
    )
    context = transform_splits_from_saved_preprocessing(
        splits,
        run_dir,
    )
    arrays = context["arrays"]
    class_names = context["class_names"]
    n_classes = context["n_classes"]

    with open(
        run_dir / "run_metadata.json",
        "r",
        encoding="utf-8",
    ) as handle:
        run_metadata = json.load(handle)

    selected_candidate = str(
        run_metadata["selected_candidate"]
    )
    best_alpha = float(
        run_metadata["best_alpha_numeric"]
    )

    xgb_model = joblib.load(
        run_dir / "xgboost_model.joblib"
    )
    stacker = joblib.load(
        run_dir / "fusion_stacker.joblib"
    )
    slm_model, slm_tokenizer = load_slm_adapter(
        run_dir,
        n_classes,
        class_names,
        run_metadata["model_checkpoint"],
    )

    probability_path = cache_dir / "probabilities.npz"

    if probability_path.exists() and CFG.posthoc_resume:
        cached = np.load(probability_path)
        probability_data = {
            key: cached[key]
            for key in cached.files
        }
        print("✅ Probabilités rechargées :", probability_path)
    else:
        probability_data = {}
        for split_name in ["cal", "test"]:
            loader = make_full_llm_loader(
                arrays,
                split_name,
                slm_tokenizer,
                context["text_features"],
                CFG,
            )
            slm_probs, labels, _ = collect_llm_probs(
                slm_model,
                loader,
                DEVICE,
            )
            xgb_probs = np.asarray(
                xgb_model.predict_proba(
                    arrays[split_name]["raw_numeric"]
                ),
                dtype=np.float64,
            )
            xgb_probs /= xgb_probs.sum(
                axis=1,
                keepdims=True,
            )
            final_probs = get_candidate_probabilities(
                selected_candidate,
                slm_probs,
                xgb_probs,
                stacker,
                best_alpha,
                n_classes,
            )
            probability_data[
                f"{split_name}_slm_probs"
            ] = slm_probs.astype(np.float32)
            probability_data[
                f"{split_name}_xgb_probs"
            ] = xgb_probs.astype(np.float32)
            probability_data[
                f"{split_name}_final_probs"
            ] = final_probs.astype(np.float32)
            probability_data[
                f"{split_name}_labels"
            ] = labels.astype(np.int64)

        np.savez_compressed(
            probability_path,
            **probability_data,
        )
        print("✅ Probabilités sauvegardées :", probability_path)

    partition = np.load(
        run_dir / "calibration_partition_indices.npz"
    )
    selection_idx = partition["selection_indices"]
    conformal_idx = partition["conformal_indices"]

    temperature_path = cache_dir / "temperature.json"
    if temperature_path.exists() and CFG.posthoc_resume:
        with open(
            temperature_path,
            "r",
            encoding="utf-8",
        ) as handle:
            calibration_info = json.load(handle)
        temperature = float(
            calibration_info["temperature"]
        )
        qhat = float(calibration_info["raps_qhat"])
    else:
        temperature = fit_temperature(
            probability_data["cal_final_probs"][
                selection_idx
            ],
            probability_data["cal_labels"][
                selection_idx
            ],
        )
        cal_probs_temp = apply_temperature(
            probability_data["cal_final_probs"][
                conformal_idx
            ],
            temperature,
        )
        cal_labels_conf = probability_data[
            "cal_labels"
        ][conformal_idx]
        scores = conformal_true_scores(
            cal_probs_temp,
            cal_labels_conf,
            method="RAPS",
            raps_lambda=CFG.raps_lambda,
            k_reg=CFG.raps_k_reg,
        )
        qhat = conformal_quantile(
            scores,
            CFG.conformal_alpha,
        )
        with open(
            temperature_path,
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                {
                    "temperature": temperature,
                    "raps_qhat": qhat,
                },
                handle,
                indent=2,
            )

    test_probs_temp = apply_temperature(
        probability_data["test_final_probs"],
        temperature,
    )
    prediction_sets = build_prediction_sets(
        test_probs_temp,
        qhat=qhat,
        method="RAPS",
        raps_lambda=CFG.raps_lambda,
        k_reg=CFG.raps_k_reg,
    )
    set_sizes = np.asarray(
        [len(values) for values in prediction_sets],
        dtype=np.int16,
    )

    return {
        "cache_dir": cache_dir,
        "run_dir": run_dir,
        "context": context,
        "arrays": arrays,
        "class_names": class_names,
        "n_classes": n_classes,
        "numeric_features": context["numeric_features"],
        "text_features": context["text_features"],
        "run_metadata": run_metadata,
        "selected_candidate": selected_candidate,
        "best_alpha": best_alpha,
        "xgb_model": xgb_model,
        "stacker": stacker,
        "slm_model": slm_model,
        "slm_tokenizer": slm_tokenizer,
        "probabilities": probability_data,
        "temperature": temperature,
        "qhat": qhat,
        "test_probs_temp": test_probs_temp,
        "prediction_sets": prediction_sets,
        "set_sizes": set_sizes,
    }

## 10 — Analyse post-hoc complète avec points de reprise

In [ ]:
all_sample_details = []
all_curve_rows = []
all_stability_rows = []
all_complementarity_rows = []
all_conformal_rows = []
all_latency_rows = []
all_verbalizer_rows = []

for _, selected_run in best_runs_df.iterrows():
    protocol = str(selected_run["protocol"])
    backbone = str(selected_run["backbone"])
    seed = int(selected_run["seed"])
    backbone_name = backbone_display_name(backbone)

    print("\n" + "=" * 100)
    print(
        f"POST-HOC XAI : {backbone_name} | "
        f"{protocol} | seed={seed}"
    )
    print("=" * 100)

    apply_backbone_config(backbone)
    bundle = scenario_probability_bundle(
        protocol,
        backbone,
        seed,
    )

    cache_dir = bundle["cache_dir"]
    arrays = bundle["arrays"]
    class_names = bundle["class_names"]
    numeric_features = bundle["numeric_features"]
    text_features = bundle["text_features"]
    n_classes = bundle["n_classes"]
    xgb_model = bundle["xgb_model"]
    stacker = bundle["stacker"]
    slm_model = bundle["slm_model"]
    slm_tokenizer = bundle["slm_tokenizer"]
    probabilities = bundle["probabilities"]
    test_probs_temp = bundle["test_probs_temp"]
    prediction_sets = bundle["prediction_sets"]
    set_sizes = bundle["set_sizes"]

    test_labels = probabilities["test_labels"]
    final_predictions = test_probs_temp.argmax(axis=1)
    xgb_predictions = probabilities[
        "test_xgb_probs"
    ].argmax(axis=1)
    slm_predictions = probabilities[
        "test_slm_probs"
    ].argmax(axis=1)

    # ----------------------------------------------------------
    # Complémentarité sur l'ensemble du test.
    # ----------------------------------------------------------
    xgb_correct = xgb_predictions == test_labels
    slm_correct = slm_predictions == test_labels
    final_correct = final_predictions == test_labels
    xgb_errors = ~xgb_correct
    slm_errors = ~slm_correct

    error_union = xgb_errors | slm_errors
    error_intersection = xgb_errors & slm_errors

    one_branch_correct = xgb_correct ^ slm_correct
    any_branch_correct = xgb_correct | slm_correct

    true_indices = np.arange(len(test_labels))
    xgb_true_prob = probabilities[
        "test_xgb_probs"
    ][true_indices, test_labels]
    slm_true_prob = probabilities[
        "test_slm_probs"
    ][true_indices, test_labels]

    complementarity_row = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "seed": seed,
        "n_test": int(len(test_labels)),
        "branch_disagreement_rate": float(
            np.mean(xgb_predictions != slm_predictions)
        ),
        "both_correct_rate": float(
            np.mean(xgb_correct & slm_correct)
        ),
        "xgb_only_correct_rate": float(
            np.mean(xgb_correct & ~slm_correct)
        ),
        "slm_only_correct_rate": float(
            np.mean(slm_correct & ~xgb_correct)
        ),
        "neither_correct_rate": float(
            np.mean(~xgb_correct & ~slm_correct)
        ),
        "one_branch_correct_rate": float(
            np.mean(one_branch_correct)
        ),
        "error_overlap_jaccard": float(
            error_intersection.sum()
            / max(1, error_union.sum())
        ),
        "oracle_accuracy": float(
            np.mean(any_branch_correct)
        ),
        "final_accuracy": float(
            np.mean(final_correct)
        ),
        "stacking_rescue_rate": float(
            np.mean(final_correct[one_branch_correct])
            if one_branch_correct.any()
            else np.nan
        ),
        "negative_transfer_rate": float(
            np.mean(~final_correct[any_branch_correct])
            if any_branch_correct.any()
            else np.nan
        ),
        "true_probability_correlation": float(
            np.corrcoef(
                xgb_true_prob,
                slm_true_prob,
            )[0, 1]
        ),
    }
    all_complementarity_rows.append(
        complementarity_row
    )

    # ----------------------------------------------------------
    # Couverture conforme complète.
    # ----------------------------------------------------------
    covered = np.asarray([
        int(label in values)
        for label, values in zip(
            test_labels,
            prediction_sets,
        )
    ])
    all_conformal_rows.append({
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "seed": seed,
        "coverage": float(covered.mean()),
        "average_set_size": float(set_sizes.mean()),
        "singleton_rate": float(
            np.mean(set_sizes == 1)
        ),
        "ambiguity_rate": float(
            np.mean(
                (set_sizes > 1)
                & (
                    set_sizes
                    <= CFG.conformal_abstain_size
                )
            )
        ),
        "abstention_rate": float(
            np.mean(
                set_sizes
                > CFG.conformal_abstain_size
            )
        ),
        "excess_coverage": float(
            covered.mean()
            - (1.0 - CFG.conformal_alpha)
        ),
        "temperature": float(bundle["temperature"]),
        "raps_qhat": float(bundle["qhat"]),
    })

    # ----------------------------------------------------------
    # Échantillons XAI.
    # ----------------------------------------------------------
    sample_index_path = (
        cache_dir / "posthoc_sample_indices.csv"
    )
    if (
        sample_index_path.exists()
        and CFG.posthoc_resume
    ):
        sample_indices = pd.read_csv(
            sample_index_path
        )["test_index"].to_numpy(dtype=np.int64)
    else:
        sample_indices = select_posthoc_indices(
            test_labels,
            final_predictions,
            CFG.posthoc_samples_per_scenario,
            CFG.posthoc_error_quota,
            seed + 1103,
        )
        atomic_csv(
            pd.DataFrame(
                {"test_index": sample_indices}
            ),
            sample_index_path,
        )

    detail_path = cache_dir / "sample_details.csv"
    curve_path = cache_dir / "deletion_curves.csv"
    stability_path = cache_dir / "stability.csv"
    verbalizer_path = cache_dir / "verbalizer.csv"

    details_df = read_csv_or_empty(detail_path)
    curves_df = read_csv_or_empty(curve_path)
    stability_df = read_csv_or_empty(stability_path)
    verbalizer_df = read_csv_or_empty(verbalizer_path)

    completed_indices = set(
        details_df["test_index"].astype(int).tolist()
        if not details_df.empty
        else []
    )
    pending_indices = [
        int(index)
        for index in sample_indices
        if int(index) not in completed_indices
    ]

    train_medians = np.median(
        arrays["train"]["raw_numeric"],
        axis=0,
    )
    train_stds = np.std(
        arrays["train"]["raw_numeric"],
        axis=0,
    )
    train_stds = np.where(
        train_stds <= 1e-12,
        1.0,
        train_stds,
    )

    baseline_rankings = {}

    if pending_indices:
        x_pending = arrays["test"][
            "raw_numeric"
        ][pending_indices]

        start_tree = time.perf_counter()
        pending_shap = get_tree_shap_values(
            xgb_model,
            x_pending,
            n_classes,
        )
        tree_seconds = time.perf_counter() - start_tree
        tree_latency_ms = (
            1000.0
            * tree_seconds
            / max(1, len(pending_indices))
        )
    else:
        pending_shap = np.empty(
            (
                0,
                n_classes,
                len(numeric_features),
            )
        )
        tree_latency_ms = np.nan

    new_detail_rows = (
        details_df.to_dict("records")
        if not details_df.empty
        else []
    )
    new_curve_rows = (
        curves_df.to_dict("records")
        if not curves_df.empty
        else []
    )

    rng = np.random.default_rng(seed + 2027)

    for local_position, test_index in enumerate(
        tqdm(
            pending_indices,
            desc=f"Post-hoc {protocol} {backbone}",
        )
    ):
        predicted_class = int(
            final_predictions[test_index]
        )
        predicted_label = class_names[
            predicted_class
        ]
        true_label = class_names[
            int(test_labels[test_index])
        ]
        confidence = float(
            test_probs_temp[
                test_index,
                predicted_class,
            ]
        )

        raw_numeric = arrays["test"][
            "raw_numeric"
        ][test_index].copy()
        text_values = arrays["test"][
            "scaled_text"
        ][test_index].copy()

        # TreeSHAP ranking.
        class_shap = pending_shap[
            local_position,
            predicted_class,
        ]
        shap_ranking = np.argsort(
            -np.abs(class_shap)
        )

        # Occlusion unitaire du SLM.
        unit_variants = np.repeat(
            text_values[None, :],
            len(text_features),
            axis=0,
        )
        unit_variants[
            np.arange(len(text_features)),
            np.arange(len(text_features)),
        ] = 0.0

        start_occ = time.perf_counter()
        unit_probs = slm_predict_values(
            slm_model,
            slm_tokenizer,
            text_features,
            unit_variants,
        )
        occ_seconds = (
            time.perf_counter() - start_occ
        )
        occ_latency_ms = (
            1000.0
            * occ_seconds
            / max(1, len(text_features))
        )

        slm_original_prob = float(
            probabilities["test_slm_probs"][
                test_index,
                predicted_class,
            ]
        )
        xgb_original_prob = float(
            probabilities["test_xgb_probs"][
                test_index,
                predicted_class,
            ]
        )

        occlusion_values = (
            slm_original_prob
            - unit_probs[:, predicted_class]
        )
        occlusion_ranking = np.argsort(
            -occlusion_values
        )

        baseline_rankings[test_index] = {
            "tree_values": class_shap.copy(),
            "tree_ranking": shap_ranking.copy(),
            "semantic_values": occlusion_values.copy(),
            "semantic_ranking": occlusion_ranking.copy(),
        }

        numeric_top_names = [
            numeric_features[index]
            for index in shap_ranking[
                :CFG.explanation_top_k
            ]
        ]
        semantic_top_names = [
            text_features[index]
            for index in occlusion_ranking[
                :CFG.explanation_top_k
            ]
        ]

        conformal_labels = [
            class_names[index]
            for index in prediction_sets[test_index]
        ]

        sample_curve_rows = []

        # Préparer toutes les variantes numériques et textuelles.
        numeric_variants = []
        numeric_meta = []
        text_variants = []
        text_meta = []

        for k in CFG.deletion_k_values:
            effective_numeric_k = min(
                k,
                len(numeric_features),
            )
            effective_text_k = min(
                k,
                len(text_features),
            )

            numeric_top = shap_ranking[
                :effective_numeric_k
            ]
            semantic_top = occlusion_ranking[
                :effective_text_k
            ]

            # Suppression Top-k.
            numeric_deleted = raw_numeric.copy()
            numeric_deleted[numeric_top] = (
                train_medians[numeric_top]
            )
            numeric_variants.append(
                numeric_deleted
            )
            numeric_meta.append(
                ("topk_delete", k, -1)
            )

            text_deleted = text_values.copy()
            text_deleted[semantic_top] = 0.0
            text_variants.append(text_deleted)
            text_meta.append(
                ("topk_delete", k, -1)
            )

            # Suffisance Top-k.
            numeric_sufficient = (
                train_medians.copy()
            )
            numeric_sufficient[numeric_top] = (
                raw_numeric[numeric_top]
            )
            numeric_variants.append(
                numeric_sufficient
            )
            numeric_meta.append(
                ("topk_keep", k, -1)
            )

            text_sufficient = np.zeros_like(
                text_values
            )
            text_sufficient[semantic_top] = (
                text_values[semantic_top]
            )
            text_variants.append(
                text_sufficient
            )
            text_meta.append(
                ("topk_keep", k, -1)
            )

            # Baseline aléatoire répétée.
            for repeat in range(
                CFG.random_baseline_repeats
            ):
                numeric_random = rng.choice(
                    len(numeric_features),
                    size=effective_numeric_k,
                    replace=False,
                )
                numeric_random_deleted = (
                    raw_numeric.copy()
                )
                numeric_random_deleted[
                    numeric_random
                ] = train_medians[numeric_random]
                numeric_variants.append(
                    numeric_random_deleted
                )
                numeric_meta.append(
                    ("random_delete", k, repeat)
                )

                semantic_random = rng.choice(
                    len(text_features),
                    size=effective_text_k,
                    replace=False,
                )
                text_random_deleted = (
                    text_values.copy()
                )
                text_random_deleted[
                    semantic_random
                ] = 0.0
                text_variants.append(
                    text_random_deleted
                )
                text_meta.append(
                    ("random_delete", k, repeat)
                )

        numeric_variant_probs = np.asarray(
            xgb_model.predict_proba(
                np.asarray(
                    numeric_variants,
                    dtype=np.float32,
                )
            ),
            dtype=np.float64,
        )
        text_variant_probs = slm_predict_values(
            slm_model,
            slm_tokenizer,
            text_features,
            np.asarray(
                text_variants,
                dtype=np.float32,
            ),
        )

        numeric_results = {}
        for meta, probability in zip(
            numeric_meta,
            numeric_variant_probs[
                :,
                predicted_class,
            ],
        ):
            numeric_results.setdefault(
                (meta[0], meta[1]),
                [],
            ).append(float(probability))

        semantic_results = {}
        for meta, probability in zip(
            text_meta,
            text_variant_probs[
                :,
                predicted_class,
            ],
        ):
            semantic_results.setdefault(
                (meta[0], meta[1]),
                [],
            ).append(float(probability))

        for branch, original_prob, results in [
            (
                "numeric_tree_shap",
                xgb_original_prob,
                numeric_results,
            ),
            (
                "semantic_occlusion",
                slm_original_prob,
                semantic_results,
            ),
        ]:
            for k in CFG.deletion_k_values:
                top_deleted_prob = results[
                    ("topk_delete", k)
                ][0]
                top_kept_prob = results[
                    ("topk_keep", k)
                ][0]
                random_probabilities = results[
                    ("random_delete", k)
                ]

                sample_curve_rows.append({
                    "protocol": protocol,
                    "backbone": backbone,
                    "backbone_display": backbone_name,
                    "seed": seed,
                    "test_index": test_index,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                    "is_correct": bool(
                        predicted_label == true_label
                    ),
                    "conformal_set_size": int(
                        set_sizes[test_index]
                    ),
                    "conformal_category": (
                        conformal_category(
                            int(set_sizes[test_index])
                        )
                    ),
                    "branch": branch,
                    "k": int(k),
                    "original_probability": original_prob,
                    "topk_deleted_probability": (
                        top_deleted_prob
                    ),
                    "topk_drop": (
                        original_prob
                        - top_deleted_prob
                    ),
                    "topk_kept_probability": (
                        top_kept_prob
                    ),
                    "sufficiency_loss": (
                        original_prob
                        - top_kept_prob
                    ),
                    "random_deleted_probability_mean": (
                        float(
                            np.mean(
                                random_probabilities
                            )
                        )
                    ),
                    "random_drop_mean": (
                        original_prob
                        - float(
                            np.mean(
                                random_probabilities
                            )
                        )
                    ),
                    "random_drop_std": float(
                        np.std(
                            [
                                original_prob - value
                                for value
                                in random_probabilities
                            ],
                            ddof=1,
                        )
                    ),
                    "xai_gain_over_random": (
                        (
                            original_prob
                            - top_deleted_prob
                        )
                        - (
                            original_prob
                            - float(
                                np.mean(
                                    random_probabilities
                                )
                            )
                        )
                    ),
                })

        new_curve_rows.extend(
            sample_curve_rows
        )

        k_reference = min(
            CFG.explanation_top_k,
            len(numeric_features),
            len(text_features),
        )
        tree_occ_jaccard = (
            len(
                set(
                    numeric_top_names[
                        :k_reference
                    ]
                )
                & set(
                    semantic_top_names[
                        :k_reference
                    ]
                )
            )
            / max(
                1,
                len(
                    set(
                        numeric_top_names[
                            :k_reference
                        ]
                    )
                    | set(
                        semantic_top_names[
                            :k_reference
                        ]
                    )
                ),
            )
        )

        new_detail_rows.append({
            "protocol": protocol,
            "backbone": backbone,
            "backbone_display": backbone_name,
            "seed": seed,
            "test_index": test_index,
            "true_label": true_label,
            "predicted_label": predicted_label,
            "is_correct": bool(
                predicted_label == true_label
            ),
            "confidence": confidence,
            "conformal_set": " | ".join(
                conformal_labels
            ),
            "conformal_set_size": int(
                set_sizes[test_index]
            ),
            "conformal_category": (
                conformal_category(
                    int(set_sizes[test_index])
                )
            ),
            "numeric_top_features": " | ".join(
                numeric_top_names
            ),
            "semantic_top_features": " | ".join(
                semantic_top_names
            ),
            "cross_branch_jaccard_top5": (
                tree_occ_jaccard
            ),
            "tree_shap_latency_ms_per_sample": (
                tree_latency_ms
            ),
            "semantic_unit_occlusion_latency_ms": (
                occ_latency_ms
            ),
            "xgb_original_probability": (
                xgb_original_prob
            ),
            "slm_original_probability": (
                slm_original_prob
            ),
        })

        if (
            len(new_detail_rows)
            % CFG.posthoc_checkpoint_every
            == 0
        ):
            atomic_csv(
                pd.DataFrame(new_detail_rows),
                detail_path,
            )
            atomic_csv(
                pd.DataFrame(new_curve_rows),
                curve_path,
            )

    details_df = pd.DataFrame(new_detail_rows)
    curves_df = pd.DataFrame(new_curve_rows)
    atomic_csv(details_df, detail_path)
    atomic_csv(curves_df, curve_path)

    # Reconstituer les rankings des échantillons déjà sauvegardés
    # uniquement si une analyse de stabilité reste à faire.
    completed_stability = set(
        zip(
            stability_df.get(
                "test_index",
                pd.Series(dtype=int),
            ).astype(int),
            stability_df.get(
                "repeat",
                pd.Series(dtype=int),
            ).astype(int),
        )
    )
    stability_candidates = sample_indices[
        :min(
            CFG.stability_samples,
            len(sample_indices),
        )
    ]

    new_stability_rows = (
        stability_df.to_dict("records")
        if not stability_df.empty
        else []
    )

    for test_index in tqdm(
        stability_candidates,
        desc=f"Stability {protocol} {backbone}",
    ):
        test_index = int(test_index)

        # Calculer/recalculer le ranking de référence.
        predicted_class = int(
            final_predictions[test_index]
        )
        raw_numeric = arrays["test"][
            "raw_numeric"
        ][test_index].copy()
        text_values = arrays["test"][
            "scaled_text"
        ][test_index].copy()

        base_shap = get_tree_shap_values(
            xgb_model,
            raw_numeric[None, :],
            n_classes,
        )[0, predicted_class]
        base_tree_rank = np.argsort(
            -np.abs(base_shap)
        )

        base_unit_variants = np.repeat(
            text_values[None, :],
            len(text_features),
            axis=0,
        )
        base_unit_variants[
            np.arange(len(text_features)),
            np.arange(len(text_features)),
        ] = 0.0
        base_unit_probs = slm_predict_values(
            slm_model,
            slm_tokenizer,
            text_features,
            base_unit_variants,
        )
        base_semantic_values = (
            probabilities["test_slm_probs"][
                test_index,
                predicted_class,
            ]
            - base_unit_probs[:, predicted_class]
        )
        base_semantic_rank = np.argsort(
            -base_semantic_values
        )

        for repeat in range(
            CFG.stability_repeats
        ):
            if (
                test_index,
                repeat,
            ) in completed_stability:
                continue

            local_rng = np.random.default_rng(
                seed
                + test_index
                + 1009 * repeat
            )

            perturbed_numeric = (
                raw_numeric
                + local_rng.normal(
                    0.0,
                    CFG.stability_noise_fraction,
                    size=len(raw_numeric),
                )
                * train_stds
            ).astype(np.float32)

            perturbed_text = (
                text_values
                + local_rng.normal(
                    0.0,
                    CFG.stability_noise_fraction,
                    size=len(text_values),
                )
            ).astype(np.float32)

            pert_xgb_prob = np.asarray(
                xgb_model.predict_proba(
                    perturbed_numeric[None, :]
                ),
                dtype=np.float64,
            )
            pert_slm_prob = slm_predict_values(
                slm_model,
                slm_tokenizer,
                text_features,
                perturbed_text[None, :],
            )
            pert_final_prob = (
                get_candidate_probabilities(
                    bundle["selected_candidate"],
                    pert_slm_prob,
                    pert_xgb_prob,
                    stacker,
                    bundle["best_alpha"],
                    n_classes,
                )
            )
            pert_prediction = int(
                pert_final_prob.argmax(axis=1)[0]
            )

            pert_shap = get_tree_shap_values(
                xgb_model,
                perturbed_numeric[None, :],
                n_classes,
            )[0, predicted_class]
            pert_tree_rank = np.argsort(
                -np.abs(pert_shap)
            )

            pert_unit_variants = np.repeat(
                perturbed_text[None, :],
                len(text_features),
                axis=0,
            )
            pert_unit_variants[
                np.arange(len(text_features)),
                np.arange(len(text_features)),
            ] = 0.0
            pert_unit_probs = slm_predict_values(
                slm_model,
                slm_tokenizer,
                text_features,
                pert_unit_variants,
            )
            pert_semantic_values = (
                float(
                    pert_slm_prob[
                        0,
                        predicted_class,
                    ]
                )
                - pert_unit_probs[
                    :,
                    predicted_class,
                ]
            )
            pert_semantic_rank = np.argsort(
                -pert_semantic_values
            )

            new_stability_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_name,
                "seed": seed,
                "test_index": test_index,
                "repeat": repeat,
                "true_label": class_names[
                    int(test_labels[test_index])
                ],
                "predicted_label": class_names[
                    predicted_class
                ],
                "class_stable": bool(
                    pert_prediction
                    == predicted_class
                ),
                "tree_jaccard_top5": (
                    jaccard_top_k(
                        base_tree_rank.tolist(),
                        pert_tree_rank.tolist(),
                        CFG.explanation_top_k,
                    )
                ),
                "semantic_jaccard_top5": (
                    jaccard_top_k(
                        base_semantic_rank.tolist(),
                        pert_semantic_rank.tolist(),
                        CFG.explanation_top_k,
                    )
                ),
                "tree_spearman": safe_spearman(
                    np.abs(base_shap),
                    np.abs(pert_shap),
                ),
                "semantic_spearman": safe_spearman(
                    base_semantic_values,
                    pert_semantic_values,
                ),
                "confidence_shift": float(
                    pert_final_prob[
                        0,
                        predicted_class,
                    ]
                    - test_probs_temp[
                        test_index,
                        predicted_class,
                    ]
                ),
                "conformal_set_size": int(
                    set_sizes[test_index]
                ),
                "conformal_category": (
                    conformal_category(
                        int(set_sizes[test_index])
                    )
                ),
            })

            if (
                len(new_stability_rows)
                % CFG.posthoc_checkpoint_every
                == 0
            ):
                atomic_csv(
                    pd.DataFrame(
                        new_stability_rows
                    ),
                    stability_path,
                )

    stability_df = pd.DataFrame(
        new_stability_rows
    )
    atomic_csv(
        stability_df,
        stability_path,
    )

    # ----------------------------------------------------------
    # Verbalisation structurée avec validation et fallback.
    # ----------------------------------------------------------
    completed_verbalizer = set(
        verbalizer_df["test_index"].astype(int).tolist()
        if not verbalizer_df.empty
        else []
    )
    verbalizer_candidates = [
        int(index)
        for index in sample_indices[
            :min(
                CFG.verbalizer_samples,
                len(sample_indices),
            )
        ]
        if int(index) not in completed_verbalizer
    ]

    generated_rows = (
        verbalizer_df.to_dict("records")
        if not verbalizer_df.empty
        else []
    )

    if verbalizer_candidates:
        detail_lookup = details_df.set_index(
            "test_index"
        )
        prompts = []
        prompt_metadata = []

        for test_index in verbalizer_candidates:
            row = detail_lookup.loc[test_index]
            numeric_names = str(
                row["numeric_top_features"]
            ).split(" | ")
            semantic_names = str(
                row["semantic_top_features"]
            ).split(" | ")
            conformal_labels = str(
                row["conformal_set"]
            ).split(" | ")

            prompt = build_structured_prompt(
                str(row["predicted_label"]),
                float(row["confidence"]),
                conformal_labels,
                numeric_names,
                semantic_names,
            )
            prompts.append(prompt)
            prompt_metadata.append({
                "test_index": test_index,
                "predicted_label": str(
                    row["predicted_label"]
                ),
                "numeric_names": numeric_names,
                "semantic_names": semantic_names,
                "conformal_labels": conformal_labels,
                "confidence": float(
                    row["confidence"]
                ),
            })

        first_outputs, first_seconds = (
            generate_slm_explanations(prompts)
        )
        first_latency = (
            first_seconds
            / max(1, len(first_outputs))
        )

        invalid_positions = []
        validation_cache = []

        for position, (
            output,
            metadata_row,
        ) in enumerate(
            zip(
                first_outputs,
                prompt_metadata,
            )
        ):
            validation = (
                validate_structured_explanation(
                    output,
                    metadata_row[
                        "predicted_label"
                    ],
                    metadata_row[
                        "numeric_names"
                    ],
                    metadata_row[
                        "semantic_names"
                    ],
                )
            )
            validation_cache.append(
                validation
            )
            if not validation["valid"]:
                invalid_positions.append(position)

        repaired_outputs = {}
        repaired_latency = 0.0

        if (
            invalid_positions
            and CFG.verbalizer_retry
        ):
            repair_prompts = []
            for position in invalid_positions:
                metadata_row = prompt_metadata[
                    position
                ]
                repair_prompts.append(
                    build_structured_prompt(
                        metadata_row[
                            "predicted_label"
                        ],
                        metadata_row[
                            "confidence"
                        ],
                        metadata_row[
                            "conformal_labels"
                        ],
                        metadata_row[
                            "numeric_names"
                        ],
                        metadata_row[
                            "semantic_names"
                        ],
                    )
                    + "\nThe previous answer was invalid. "
                    "Copy the five required lines exactly."
                )

            repair_values, repair_seconds = (
                generate_slm_explanations(
                    repair_prompts
                )
            )
            repaired_latency = (
                repair_seconds
                / max(1, len(repair_values))
            )
            repaired_outputs = dict(
                zip(
                    invalid_positions,
                    repair_values,
                )
            )

        for position, metadata_row in enumerate(
            prompt_metadata
        ):
            first_text = first_outputs[position]
            first_validation = (
                validation_cache[position]
            )

            final_text = first_text
            status = "accepted_first_pass"
            latency_seconds = first_latency

            if not first_validation["valid"]:
                repaired_text = repaired_outputs.get(
                    position
                )
                if repaired_text is not None:
                    repaired_validation = (
                        validate_structured_explanation(
                            repaired_text,
                            metadata_row[
                                "predicted_label"
                            ],
                            metadata_row[
                                "numeric_names"
                            ],
                            metadata_row[
                                "semantic_names"
                            ],
                        )
                    )
                else:
                    repaired_validation = {
                        "valid": False
                    }

                if repaired_validation.get(
                    "valid",
                    False,
                ):
                    final_text = repaired_text
                    status = "accepted_after_repair"
                    latency_seconds += (
                        repaired_latency
                    )
                    final_validation = (
                        repaired_validation
                    )
                else:
                    final_text = (
                        deterministic_template_explanation(
                            metadata_row[
                                "predicted_label"
                            ],
                            metadata_row[
                                "confidence"
                            ],
                            metadata_row[
                                "conformal_labels"
                            ],
                            [
                                (name, np.nan)
                                for name in metadata_row[
                                    "numeric_names"
                                ]
                            ],
                            [
                                (name, np.nan)
                                for name in metadata_row[
                                    "semantic_names"
                                ]
                            ],
                        )
                    )
                    status = (
                        "deterministic_fallback"
                    )
                    final_validation = (
                        validate_structured_explanation(
                            build_structured_prompt(
                                metadata_row[
                                    "predicted_label"
                                ],
                                metadata_row[
                                    "confidence"
                                ],
                                metadata_row[
                                    "conformal_labels"
                                ],
                                metadata_row[
                                    "numeric_names"
                                ],
                                metadata_row[
                                    "semantic_names"
                                ],
                            ),
                            metadata_row[
                                "predicted_label"
                            ],
                            metadata_row[
                                "numeric_names"
                            ],
                            metadata_row[
                                "semantic_names"
                            ],
                        )
                    )
            else:
                final_validation = (
                    first_validation
                )

            citation_result = citation_metrics(
                final_text,
                metadata_row[
                    "numeric_names"
                ]
                + metadata_row[
                    "semantic_names"
                ],
            )

            generated_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_name,
                "seed": seed,
                "test_index": metadata_row[
                    "test_index"
                ],
                "status": status,
                "text": final_text,
                "latency_seconds": (
                    latency_seconds
                ),
                "word_count": len(
                    str(final_text).split()
                ),
                **{
                    key: value
                    for key, value
                    in final_validation.items()
                    if key != "valid"
                },
                "valid_final": bool(
                    final_validation.get(
                        "valid",
                        status
                        == "deterministic_fallback",
                    )
                ),
                **citation_result,
            })

        verbalizer_df = pd.DataFrame(
            generated_rows
        )
        atomic_csv(
            verbalizer_df,
            verbalizer_path,
        )

    # ----------------------------------------------------------
    # Collecte globale.
    # ----------------------------------------------------------
    details_df = read_csv_or_empty(
        detail_path
    )
    curves_df = read_csv_or_empty(
        curve_path
    )
    stability_df = read_csv_or_empty(
        stability_path
    )
    verbalizer_df = read_csv_or_empty(
        verbalizer_path
    )

    all_sample_details.append(details_df)
    all_curve_rows.append(curves_df)
    all_stability_rows.append(stability_df)
    all_verbalizer_rows.append(verbalizer_df)

    all_latency_rows.extend([
        {
            "protocol": protocol,
            "backbone": backbone,
            "backbone_display": backbone_name,
            "method": "TreeSHAP",
            "latency_ms": float(
                details_df[
                    "tree_shap_latency_ms_per_sample"
                ].mean()
            ),
        },
        {
            "protocol": protocol,
            "backbone": backbone,
            "backbone_display": backbone_name,
            "method": "SLM feature occlusion",
            "latency_ms": float(
                details_df[
                    "semantic_unit_occlusion_latency_ms"
                ].mean()
            ),
        },
    ])

    del bundle
    del slm_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

sample_details_df = pd.concat(
    all_sample_details,
    ignore_index=True,
)
deletion_curves_df = pd.concat(
    all_curve_rows,
    ignore_index=True,
)
stability_results_df = pd.concat(
    all_stability_rows,
    ignore_index=True,
)
verbalizer_results_df = pd.concat(
    all_verbalizer_rows,
    ignore_index=True,
)
complementarity_df = pd.DataFrame(
    all_complementarity_rows
)
conformal_posthoc_df = pd.DataFrame(
    all_conformal_rows
)
xai_latency_df = pd.DataFrame(
    all_latency_rows
)

atomic_csv(
    sample_details_df,
    POSTHOC_TABLE_DIR / "posthoc_sample_details.csv",
)
atomic_csv(
    deletion_curves_df,
    POSTHOC_TABLE_DIR / "deletion_random_sufficiency_curves.csv",
)
atomic_csv(
    stability_results_df,
    POSTHOC_TABLE_DIR / "explanation_stability.csv",
)
atomic_csv(
    verbalizer_results_df,
    POSTHOC_TABLE_DIR / "controlled_verbalizer_results.csv",
)
atomic_csv(
    complementarity_df,
    POSTHOC_TABLE_DIR / "branch_complementarity.csv",
)
atomic_csv(
    conformal_posthoc_df,
    POSTHOC_TABLE_DIR / "posthoc_conformal_summary.csv",
)
atomic_csv(
    xai_latency_df,
    POSTHOC_TABLE_DIR / "posthoc_xai_latency.csv",
)

print("✅ Analyse post-hoc complète.")

## 11 — Agrégations, AOPC, tests statistiques et analyses par classe

In [ ]:
# AOPC : moyenne des baisses Top-k à travers les valeurs de k.
aopc_df = (
    deletion_curves_df
    .groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "seed",
        "test_index",
        "true_label",
        "predicted_label",
        "is_correct",
        "conformal_category",
        "branch",
    ], as_index=False)
    .agg(
        aopc=("topk_drop", "mean"),
        random_aopc=("random_drop_mean", "mean"),
        xai_gain_over_random=(
            "xai_gain_over_random",
            "mean",
        ),
        mean_sufficiency_loss=(
            "sufficiency_loss",
            "mean",
        ),
    )
)

# Fidélité par classe.
per_class_xai_df = (
    aopc_df
    .groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "branch",
        "true_label",
    ], as_index=False)
    .agg(
        n=("test_index", "count"),
        aopc_mean=("aopc", "mean"),
        aopc_std=("aopc", "std"),
        random_aopc_mean=(
            "random_aopc",
            "mean",
        ),
        gain_over_random_mean=(
            "xai_gain_over_random",
            "mean",
        ),
        sufficiency_loss_mean=(
            "mean_sufficiency_loss",
            "mean",
        ),
    )
)

# Correct vs erreurs.
error_xai_df = (
    aopc_df
    .groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "branch",
        "is_correct",
    ], as_index=False)
    .agg(
        n=("test_index", "count"),
        aopc_mean=("aopc", "mean"),
        gain_over_random_mean=(
            "xai_gain_over_random",
            "mean",
        ),
        sufficiency_loss_mean=(
            "mean_sufficiency_loss",
            "mean",
        ),
    )
)

# Relation entre incertitude conforme et XAI.
conformal_xai_df = (
    aopc_df
    .groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "branch",
        "conformal_category",
    ], as_index=False)
    .agg(
        n=("test_index", "count"),
        aopc_mean=("aopc", "mean"),
        gain_over_random_mean=(
            "xai_gain_over_random",
            "mean",
        ),
        sufficiency_loss_mean=(
            "mean_sufficiency_loss",
            "mean",
        ),
    )
)

stability_summary_df = (
    stability_results_df
    .groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "conformal_category",
    ], as_index=False)
    .agg(
        n=("test_index", "count"),
        class_stability_rate=(
            "class_stable",
            "mean",
        ),
        tree_jaccard_mean=(
            "tree_jaccard_top5",
            "mean",
        ),
        semantic_jaccard_mean=(
            "semantic_jaccard_top5",
            "mean",
        ),
        tree_spearman_mean=(
            "tree_spearman",
            "mean",
        ),
        semantic_spearman_mean=(
            "semantic_spearman",
            "mean",
        ),
    )
)

# Wilcoxon Top-k contre random-k.
statistical_rows = []

if wilcoxon is not None:
    for group_values, subset in deletion_curves_df.groupby([
        "protocol",
        "backbone",
        "backbone_display",
        "branch",
        "k",
    ]):
        paired = subset[[
            "topk_drop",
            "random_drop_mean",
        ]].dropna()

        if len(paired) >= 5:
            try:
                statistic, pvalue = wilcoxon(
                    paired["topk_drop"],
                    paired["random_drop_mean"],
                    alternative="greater",
                )
            except Exception:
                statistic, pvalue = np.nan, np.nan
        else:
            statistic, pvalue = np.nan, np.nan

        statistical_rows.append({
            "protocol": group_values[0],
            "backbone": group_values[1],
            "backbone_display": group_values[2],
            "branch": group_values[3],
            "k": group_values[4],
            "n_pairs": len(paired),
            "topk_drop_mean": paired[
                "topk_drop"
            ].mean(),
            "random_drop_mean": paired[
                "random_drop_mean"
            ].mean(),
            "mean_effect": (
                paired["topk_drop"]
                - paired["random_drop_mean"]
            ).mean(),
            "wilcoxon_statistic": statistic,
            "wilcoxon_pvalue_one_sided": pvalue,
        })

xai_statistical_tests_df = pd.DataFrame(
    statistical_rows
)

for frame, filename in [
    (
        aopc_df,
        "aopc_per_sample.csv",
    ),
    (
        per_class_xai_df,
        "xai_by_class.csv",
    ),
    (
        error_xai_df,
        "xai_correct_vs_errors.csv",
    ),
    (
        conformal_xai_df,
        "xai_by_conformal_category.csv",
    ),
    (
        stability_summary_df,
        "stability_by_conformal_category.csv",
    ),
    (
        xai_statistical_tests_df,
        "xai_topk_vs_random_tests.csv",
    ),
]:
    atomic_csv(
        frame,
        POSTHOC_TABLE_DIR / filename,
    )

display(per_class_xai_df)
display(error_xai_df)
display(conformal_xai_df)
display(stability_summary_df)
display(xai_statistical_tests_df)

## 12 — Graphiques article-ready

In [ ]:
# 1. Courbes de suppression Top-k vs Random-k.
curve_summary = (
    deletion_curves_df
    .groupby([
        "protocol",
        "backbone_display",
        "branch",
        "k",
    ], as_index=False)
    .agg(
        topk_drop_mean=("topk_drop", "mean"),
        topk_drop_std=("topk_drop", "std"),
        random_drop_mean=(
            "random_drop_mean",
            "mean",
        ),
    )
)

for (
    protocol,
    backbone_name,
), subset_scenario in curve_summary.groupby([
    "protocol",
    "backbone_display",
]):
    plt.figure(figsize=(10, 6))
    for branch, subset in subset_scenario.groupby(
        "branch"
    ):
        plt.plot(
            subset["k"],
            subset["topk_drop_mean"],
            marker="o",
            label=f"{branch} — Top-k",
        )
        plt.plot(
            subset["k"],
            subset["random_drop_mean"],
            marker="x",
            linestyle="--",
            label=f"{branch} — Random-k",
        )
    plt.xlabel("Nombre de caractéristiques supprimées (k)")
    plt.ylabel("Baisse de probabilité")
    plt.title(
        f"Suppression Top-k vs aléatoire — "
        f"{backbone_name} | {protocol}"
    )
    plt.legend()
    save_posthoc_figure(
        f"deletion_topk_random_{protocol}_{backbone_name.lower()}"
    )
    plt.show()

# 2. Suffisance.
sufficiency_summary = (
    deletion_curves_df
    .groupby([
        "protocol",
        "backbone_display",
        "branch",
        "k",
    ], as_index=False)["sufficiency_loss"]
    .mean()
)

for (
    protocol,
    backbone_name,
), subset_scenario in sufficiency_summary.groupby([
    "protocol",
    "backbone_display",
]):
    plt.figure(figsize=(9, 5.5))
    for branch, subset in subset_scenario.groupby(
        "branch"
    ):
        plt.plot(
            subset["k"],
            subset["sufficiency_loss"],
            marker="o",
            label=branch,
        )
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Nombre de caractéristiques conservées (k)")
    plt.ylabel("Perte de suffisance")
    plt.title(
        f"Suffisance des preuves — "
        f"{backbone_name} | {protocol}"
    )
    plt.legend()
    save_posthoc_figure(
        f"sufficiency_{protocol}_{backbone_name.lower()}"
    )
    plt.show()

# 3. AOPC par classe.
for (
    protocol,
    branch,
), subset in per_class_xai_df.groupby([
    "protocol",
    "branch",
]):
    pivot = subset.pivot(
        index="true_label",
        columns="backbone_display",
        values="aopc_mean",
    )
    plt.figure(figsize=(9, 6))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
    )
    plt.title(
        f"AOPC par classe — {branch} | {protocol}"
    )
    plt.ylabel("Classe réelle")
    save_posthoc_figure(
        f"aopc_by_class_{protocol}_{branch}"
    )
    plt.show()

# 4. Stabilité des explications.
stability_long = stability_summary_df.melt(
    id_vars=[
        "protocol",
        "backbone_display",
        "conformal_category",
    ],
    value_vars=[
        "class_stability_rate",
        "tree_jaccard_mean",
        "semantic_jaccard_mean",
    ],
    var_name="metric",
    value_name="value",
)

for protocol, subset in stability_long.groupby(
    "protocol"
):
    labels = (
        subset["backbone_display"]
        + "\n"
        + subset["conformal_category"]
        + "\n"
        + subset["metric"]
    )
    plt.figure(figsize=(14, 6))
    plt.bar(labels, subset["value"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("Score")
    plt.title(
        f"Stabilité des décisions et preuves — {protocol}"
    )
    plt.xticks(rotation=35, ha="right")
    save_posthoc_figure(
        f"stability_{protocol}"
    )
    plt.show()

# 5. Complémentarité des branches.
complementarity_metrics = [
    "branch_disagreement_rate",
    "xgb_only_correct_rate",
    "slm_only_correct_rate",
    "stacking_rescue_rate",
    "negative_transfer_rate",
]

comp_long = complementarity_df.melt(
    id_vars=[
        "protocol",
        "backbone_display",
    ],
    value_vars=complementarity_metrics,
    var_name="metric",
    value_name="value",
)

for protocol, subset in comp_long.groupby(
    "protocol"
):
    labels = (
        subset["backbone_display"]
        + "\n"
        + subset["metric"]
    )
    plt.figure(figsize=(13, 6))
    plt.bar(labels, subset["value"])
    plt.ylabel("Taux")
    plt.title(
        f"Complémentarité XGBoost–SLM — {protocol}"
    )
    plt.xticks(rotation=35, ha="right")
    save_posthoc_figure(
        f"branch_complementarity_{protocol}"
    )
    plt.show()

# 6. Fidélité selon l'incertitude conforme.
for (
    protocol,
    branch,
), subset in conformal_xai_df.groupby([
    "protocol",
    "branch",
]):
    labels = (
        subset["backbone_display"]
        + "\n"
        + subset["conformal_category"]
    )
    plt.figure(figsize=(10, 5.5))
    plt.bar(
        labels,
        subset["gain_over_random_mean"],
    )
    plt.axhline(0.0, linestyle="--")
    plt.ylabel("AOPC Top-k − AOPC Random-k")
    plt.title(
        f"Qualité des preuves selon l'incertitude — "
        f"{branch} | {protocol}"
    )
    plt.xticks(rotation=25, ha="right")
    save_posthoc_figure(
        f"xai_by_uncertainty_{protocol}_{branch}"
    )
    plt.show()

# 7. Latence explicative.
plt.figure(figsize=(11, 6))
latency_labels = (
    xai_latency_df["backbone_display"]
    + "\n"
    + xai_latency_df["protocol"]
    + "\n"
    + xai_latency_df["method"]
)
plt.bar(
    latency_labels,
    xai_latency_df["latency_ms"],
)
plt.ylabel("Latence moyenne (ms)")
plt.title("Latence des méthodes d'explication")
plt.xticks(rotation=30, ha="right")
save_posthoc_figure("xai_latency_comparison")
plt.show()

# 8. Donuts du verbaliseur contrôlé.
if not verbalizer_results_df.empty:
    scenarios = list(
        verbalizer_results_df.groupby([
            "protocol",
            "backbone_display",
        ])
    )
    fig, axes = plt.subplots(
        1,
        len(scenarios),
        figsize=(6 * len(scenarios), 5),
    )
    if len(scenarios) == 1:
        axes = [axes]

    for ax, (
        (protocol, backbone_name),
        subset,
    ) in zip(axes, scenarios):
        counts = subset["status"].value_counts()
        ax.pie(
            counts.values,
            labels=counts.index,
            startangle=90,
            wedgeprops={
                "width": 0.42,
                "edgecolor": "white",
            },
            autopct="%1.1f%%",
        )
        ax.set_title(
            f"{backbone_name} | {protocol}"
        )

    plt.tight_layout()
    save_posthoc_figure(
        "controlled_verbalizer_status"
    )
    plt.show()

## 13 — Rapport final automatique

In [ ]:
final_summary = {
    "detector_training_performed": False,
    "runs_reloaded": int(len(summaries_df)),
    "scenarios_analyzed": int(
        len(best_runs_df)
    ),
    "samples_explained": int(
        sample_details_df["test_index"].nunique()
    ),
    "deletion_k_values": (
        CFG.deletion_k_values
    ),
    "random_baseline_repeats": int(
        CFG.random_baseline_repeats
    ),
    "stability_repeats": int(
        CFG.stability_repeats
    ),
    "output_directory": str(
        POSTHOC_DIR
    ),
}

with open(
    POSTHOC_TABLE_DIR
    / "posthoc_explainability_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        final_summary,
        handle,
        indent=2,
        ensure_ascii=False,
    )

best_xai_gain = (
    aopc_df
    .groupby([
        "protocol",
        "backbone_display",
        "branch",
    ], as_index=False)[
        "xai_gain_over_random"
    ]
    .mean()
    .sort_values(
        "xai_gain_over_random",
        ascending=False,
    )
)

verbalizer_summary = (
    verbalizer_results_df
    .groupby([
        "protocol",
        "backbone_display",
        "status",
    ], as_index=False)
    .agg(
        n=("test_index", "count"),
        valid_rate=("valid_final", "mean"),
        citation_validity=(
            "valid_citation_rate",
            "mean",
        ),
        unsupported_rate=(
            "unsupported_citation_rate",
            "mean",
        ),
        latency_seconds=(
            "latency_seconds",
            "mean",
        ),
    )
    if not verbalizer_results_df.empty
    else pd.DataFrame()
)

atomic_csv(
    best_xai_gain,
    POSTHOC_TABLE_DIR
    / "best_xai_gain_summary.csv",
)
if not verbalizer_summary.empty:
    atomic_csv(
        verbalizer_summary,
        POSTHOC_TABLE_DIR
        / "controlled_verbalizer_summary.csv",
    )

display(best_xai_gain)
display(complementarity_df)
display(conformal_posthoc_df)
display(verbalizer_summary)

print("\n" + "=" * 80)
print("PIPELINE POST-HOC TERMINÉ")
print("=" * 80)
print("Aucun détecteur n'a été réentraîné.")
print("Tables :", POSTHOC_TABLE_DIR)
print("Figures :", POSTHOC_FIG_DIR)
print("Caches de reprise :", POSTHOC_CACHE_DIR)

## 14 — Interprétation attendue

Les résultats doivent être interprétés selon les critères suivants :

1. **Top-k > Random-k** : les caractéristiques sélectionnées sont plus
   informatives qu'une perturbation arbitraire.
2. **AOPC élevé** : la suppression progressive des preuves réduit fortement
   la probabilité de la classe expliquée.
3. **Faible perte de suffisance** : les preuves principales conservent une
   part importante de la décision.
4. **Stabilité élevée** : de petites perturbations ne changent pas
   arbitrairement le classement des preuves.
5. **Complémentarité réelle** : une branche corrige des erreurs commises par
   l'autre, et le stacking exploite ces désaccords.
6. **Relation XAI–conformité** : les cas ambigus ou d'abstention devraient
   présenter des preuves moins stables ou moins discriminantes.
7. **Verbalisation contrôlée** : toute sortie invalide est réparée ou remplacée
   par un gabarit déterministe, sans modifier la prédiction.

## ABLATION : WITH VS WITHOUT SEMANTICS

In [ ]:
# ============================================================
# ABLATION : WITH VS WITHOUT SEMANTICS
# ============================================================

branch_long = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
)

# Harmonisation des noms des branches SLM.
branch_long["configuration"] = np.where(
    branch_long["branch"] == "XGBoost",
    "Without semantics",
    np.where(
        branch_long["branch"] == "Final",
        "With semantics",
        "Semantic only",
    ),
)

metrics = [
    "macro_f1",
    "macro_f1_all",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
]

# Tableau agrégé.
semantic_ablation_summary = (
    branch_long
    .groupby([
        "protocol",
        "backbone",
        "configuration",
    ])[metrics]
    .agg(["mean", "std"])
)

semantic_ablation_summary.to_csv(
    POSTHOC_TABLE_DIR / "with_without_semantics_summary.csv"
)

display(semantic_ablation_summary)

# ------------------------------------------------------------
# Différences appariées : hybride - XGBoost
# ------------------------------------------------------------

gain_rows = []

for protocol in branch_long["protocol"].unique():
    for backbone in branch_long["backbone"].unique():
        subset = branch_long[
            (branch_long["protocol"] == protocol)
            & (branch_long["backbone"] == backbone)
        ]

        for metric in metrics:
            pivot = subset.pivot(
                index="seed",
                columns="configuration",
                values=metric,
            )

            required = {
                "With semantics",
                "Without semantics",
            }

            if not required.issubset(pivot.columns):
                continue

            paired = pivot[
                ["With semantics", "Without semantics"]
            ].dropna()

            differences = (
                paired["With semantics"]
                - paired["Without semantics"]
            )

            if wilcoxon is not None and len(paired) >= 2:
                try:
                    statistic, pvalue = wilcoxon(
                        paired["With semantics"],
                        paired["Without semantics"],
                        alternative="two-sided",
                    )
                except Exception:
                    statistic, pvalue = np.nan, np.nan
            else:
                statistic, pvalue = np.nan, np.nan

            gain_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "metric": metric,
                "n_seeds": len(paired),
                "without_semantics_mean": (
                    paired["Without semantics"].mean()
                ),
                "with_semantics_mean": (
                    paired["With semantics"].mean()
                ),
                "semantic_gain_mean": differences.mean(),
                "semantic_gain_std": differences.std(ddof=1),
                "wilcoxon_statistic": statistic,
                "wilcoxon_pvalue": pvalue,
            })

semantic_gain_df = pd.DataFrame(gain_rows)

semantic_gain_df.to_csv(
    POSTHOC_TABLE_DIR / "semantic_contribution_gains.csv",
    index=False,
)

display(semantic_gain_df)

# ------------------------------------------------------------
# Figure : gain de la branche sémantique
# ------------------------------------------------------------

plot_gain = semantic_gain_df[
    semantic_gain_df["metric"].isin([
        "macro_f1",
        "macro_f1_all",
        "mcc",
    ])
].copy()

plot_gain["scenario"] = (
    plot_gain["protocol"]
    + "\n"
    + plot_gain["backbone"].map(backbone_display_name)
)

for metric in plot_gain["metric"].unique():
    subset = plot_gain[
        plot_gain["metric"] == metric
    ]

    plt.figure(figsize=(10, 5.5))
    plt.bar(
        subset["scenario"],
        100.0 * subset["semantic_gain_mean"],
        yerr=100.0 * subset["semantic_gain_std"].fillna(0.0),
        capsize=4,
    )
    plt.axhline(0.0, linestyle="--")
    plt.ylabel("Gain avec sémantique (points de pourcentage)")
    plt.title(
        f"Contribution de la branche sémantique — {metric}"
    )
    plt.xticks(rotation=15)
    save_posthoc_figure(
        f"with_without_semantics_{metric}"
    )
    plt.show()